## Encoder Training in cell 3_f

## 1 · Setup

In [1]:
# Internet-first setup. With Kaggle Internet ON this installs the runtime bits
# previously prepared by the loader notebooks.
import socket, glob, os

def has_internet(host="pypi.org", port=443, timeout=3):
    try:
        socket.setdefaulttimeout(timeout)
        socket.create_connection((host, port)).close()
        return True
    except OSError:
        return False

ONLINE = has_internet()
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
_wheel_dirs = glob.glob("/kaggle/input/**/wheels", recursive=True)
if _wheel_dirs:
    get_ipython().system(f'pip install -q --no-index --find-links={_wheel_dirs[0]} bitsandbytes')
    if ONLINE:
        get_ipython().system('pip install -q -U "transformers>=4.50" accelerate hf_transfer datasets huggingface_hub kagglehub')
elif ONLINE:
    get_ipython().system('pip install -q -U "transformers>=4.50" accelerate "bitsandbytes>=0.46.1" hf_transfer datasets huggingface_hub kagglehub')
else:
    print("no wheels attached, no internet - relying on the pre-installed image")

if ONLINE:
    # csebuetnlp normalizer - BanglaBERT preprocessing best practice (fail-soft)
    get_ipython().system('pip install -q "git+https://github.com/csebuetnlp/normalizer" || true')
import bitsandbytes as _bnb_check
print("bitsandbytes:", _bnb_check.__version__, "| online:", ONLINE)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 99.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 35.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is

In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import re, json, glob, time, math, unicodedata, shutil, ast
import numpy as np
import pandas as pd
import torch
import transformers
from packaging.version import Version

print("transformers:", transformers.__version__)
print("torch       :", torch.__version__)
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}     :", torch.cuda.get_device_name(i))

assert Version(transformers.__version__) >= Version("4.50"), "Gemma 3 needs transformers >= 4.50"
assert torch.cuda.is_available(), "Enable a GPU accelerator"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:298: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


transformers: 5.14.1
torch       : 2.10.0+cu128
GPU 0     : Tesla T4
GPU 1     : Tesla T4


In [3]:
# ---- Configuration ---------------------------------------------------------
MAX_CTX_CHARS  = 777
BATCH_SIZE     = 4      # v5: OOM-free on 2xT4; auto-backoff handles OOM
MAX_LEN        = 2048
CHUNK_SIZE     = 200
USE_MATH_COT   = True   # v6.18 A/B: OFF - math rows = pure Qwen zero-shot (p_q > 0.5)
                         # + sympy only. BenHalluEval: Qwen zero-shot 26.80 BHS beat
                         # Qwen+CoT 28.90 on Reasoning. Flip back to True to restore CoT.
USE_SQUAD_REFS = True
USE_SYMPY      = True
USE_QA_VERIFY  = False  # v5.1: labeled gate invalid - QA model trained on squad_bn train split
USE_RETRIEVAL  = True
USE_ENSEMBLE   = True    # Qwen Bengali judge; auto-disabled on single GPU for 16GB safety
USE_GRAMMAR    = True    # optional: uses local/Kaggle files if present, otherwise disables cleanly
USE_REL_ROUTE  = True
USE_CTX_MATCH  = True
USE_CB_REFS    = True    # optional: uses local/Kaggle files if present, otherwise disables cleanly
USE_EXTRA_QA_REFS = True # BanglaRQA/BanglaQuAD/NCTB-QA exact QA references
USE_KAGGLE_BAGDHARA = False # off: Kaggle cannot attach new datasets in a committed
                            # (non-interactive) run; the HF bagdhara set is used instead
USE_GROUPS     = True

# v6 additions
USE_CLF        = True    # encoder classifier fine-tuned on synthetic squad_bn contrastive pairs
USE_XLING_EN   = True    # English-instruction Qwen logit pass (organizers: strongest signal)
USE_XLING_COT  = False   # v6.19 A/B: translate-verify generation OFF. Gate passed only
                         # marginally (+0.054 on 55 compass rows = noise-tier) at 30-60 min
                         # test cost, and its turning ON this run confounds the math A/B.
                         # USE_XLING_EN (cheap English-logit p_qe) stays ON. Flip to True to restore.    # translate-then-verify generation on uncertain rows (budget-capped)
XLING_COT_CAP  = 450
XLING_VAL_N    = 60
MATH_SC_N      = 1
QWEN_ONLY_MATH = True    # v6.18: math rows are decided by Qwen alone. BenHalluEval
                         # measured TigerLLM as the WORST math judge of 7 models
                         # (BHS 53.45, 85.5% miss rate) while Qwen was among the best;
                         # the closed meta-judge (80% Tiger) is also no longer fitted
                         # on math rows (v6.17). Base verdict: p_q > 0.5 (uncalibrated
                         # 0.5 - only 13 labeled math rows, too few to fit honestly);
                         # sympy still outranks it, Qwen-EN CoT still overrides, and
                         # the Tiger Bengali-CoT fallback is disabled for math.
CLF_EPOCHS, CLF_MAX_LEN, CLF_BATCH, CLF_LR, CLF_MAX_TRAIN = 2, 512, 32, 2e-5, 50000
CLF_CTX_DROP   = 0.0     # v6.21: share of encoder-training pairs with context DROPPED
                         # (teaches closed-book). p_clf is only USED on grounded, so set
                         # 0.0 to train grounded-only. A/B: compare printed grounded AUC.
CLF_SAVE_NAME = "clf_finetuned"
USE_VAL_SCORE_CACHE = False # v6.19: OFF while experimenting -> validation always re-scores fresh.
                            # Set True + attach val_scores.json to skip the ~31 min pass-1 scoring
                            # once the config is frozen (fitting/gates always re-run either way).
                            # of pass-1 scoring on reruns; the FITTING (weights/thresholds/
                            # gates) still re-runs live, so bank/tier/threshold changes are
                            # always respected. Guard invalidates on prompt/model/ctx change.  # v6.14: save/reuse the fine-tuned encoder (skips ~18 min + freezes p_clf)
MATH_COT_TOKENS = 256    # v6.2: was 320; batched generation makes CoT ~7x faster
GEN_BATCH       = 8      # prompts per qwen.generate call (auto-halves on OOM)
USE_BENHALLU    = True   # v6.2: embedded BenHalluEval refs + exact-pair tiers + real clf pairs

# Hub resources from the loader notebooks. These are open-weight / public data.
TIGER_ID = "md-nishat-008/TigerLLM-9B-it"
QWEN_ID  = "Qwen/Qwen3.5-9B"
CLF_ID   = "csebuetnlp/banglabert"
SQUAD_URL = "https://huggingface.co/datasets/csebuetnlp/squad_bn/resolve/main/data/squad_bn.tar.bz2"
WIKI_DATASET = ("wikimedia/wikipedia", "20231101.bn")
GRAMMAR_REPOS = {
    "ek_kothay_prokash": "ishtiakmoin/bangla-ek-kothay-prokash",
    "shomash": "ishtiakmoin/bangla-shomash",
    "bagdhara": "ishtiakmoin/bangla-bagdhara",
}
CB_REF_REPOS = {
    # "bcs_qs": removed on request (azminetoushikwasi/bangla-bcs-qs) - noisy
    "bangla_math": "kawchar85/Bangla-Math",
}
# Recommended online additions from the Bangla NLP catalog / challenge discussion.
# Exact repo IDs are used where known; search specs fail softly if HF mirrors are unavailable.
EXTRA_QA_REPOS = {
    "banglarqa": "sartajekram/BanglaRQA",       # answerable + unanswerable Bengali RQA
    "nctb_qa":   "ShihabReza/nctb-qa",          # 12k NCTB textbook QA (Phase-2 source)
}
EXTRA_QA_SEARCH = {
    "banglaquad": {"queries": ["BanglaQuAD", "Bangla QuAD Bengali"], "must": ["banglaquad"]},
}
CB_REF_SEARCH = {
    "bluck": {"queries": ["BLUCK Bengali", "Bangla linguistic cultural knowledge BLUCK"], "must": ["bluck"]},
}
KAGGLE_GRAMMAR_DATASETS = {
    "kaggle_bagdhara": "sakhadib/bagdhara-bangla-idioms-dataset",
}

# Set True if you explicitly want full HF snapshots copied into /kaggle/working first.
# False avoids an extra copy; AutoModel/AutoTokenizer still download from the Hub cache on demand.
DOWNLOAD_MODEL_SNAPSHOTS = False
ALLOW_QWEN_ON_SINGLE_GPU = False

# grammar dataset column names - leave None to auto-detect, set if the guess is wrong:
GRAMMAR_TERM_COL = None
GRAMMAR_ANS_COL  = None
GRAMMAR_PATH_RE  = r"bagdhara|baghdara|shomash|somash|prokash|idiom|grammar|byakoron|bakko|phrase"

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
RESOURCE_DIR = os.path.join(WORK_DIR, "olikbochon_resources")
os.makedirs(RESOURCE_DIR, exist_ok=True)
CHECKPOINT_DIR = os.path.join(WORK_DIR, "checkpoints_v6_online")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

if torch.cuda.device_count() < 2 and not ALLOW_QWEN_ON_SINGLE_GPU:
    if USE_ENSEMBLE or USE_XLING_EN or USE_XLING_COT:
        print("single GPU detected -> Qwen judge/xling passes disabled for 16GB VRAM safety")
    USE_ENSEMBLE = False
    USE_XLING_EN = False
    USE_XLING_COT = False

NEED_QWEN = USE_ENSEMBLE or USE_XLING_EN or USE_XLING_COT

def find_file(*names):
    roots = ["/kaggle/input", "/kaggle/working", "Dataset", "."]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            for name in names:
                if name in filenames:
                    return os.path.join(dirpath, name)
    return None

LABELED_PATH = find_file("dataset samples.json", "dataset_samples.json")
TEST_PATH    = find_file("test set.csv", "test_set.csv")
SAMPLE_PATH  = find_file("sample submission.csv", "sample_submission.csv")
print("labeled :", LABELED_PATH)
print("test    :", TEST_PATH)
print("sample  :", SAMPLE_PATH)
assert LABELED_PATH and TEST_PATH and SAMPLE_PATH, "competition data files not found in /kaggle/input or working directory"

def maybe_snapshot(repo_id, local_name, needed=True):
    if not needed:
        return None
    if not DOWNLOAD_MODEL_SNAPSHOTS:
        return repo_id
    out = os.path.join(RESOURCE_DIR, "models", local_name)
    if os.path.exists(os.path.join(out, "config.json")):
        return out
    assert ONLINE, f"Internet is required to download {repo_id}"
    from huggingface_hub import snapshot_download
    os.makedirs(out, exist_ok=True)
    print(f"downloading snapshot {repo_id} -> {out}")
    snapshot_download(repo_id, local_dir=out, local_dir_use_symlinks=False, resume_download=True)
    return out

def ensure_squad_tarball():
    p = find_file("squad_bn.tar.bz2")
    if p:
        return p
    if not (USE_SQUAD_REFS or USE_CTX_MATCH or USE_CLF):
        return None
    assert ONLINE, "Internet is required to download squad_bn.tar.bz2"
    import urllib.request
    out_dir = os.path.join(RESOURCE_DIR, "refs")
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, "squad_bn.tar.bz2")
    if not os.path.exists(out):
        print("downloading squad_bn (~8 MB)...")
        urllib.request.urlretrieve(SQUAD_URL, out)
    print("squad_bn tarball:", out, f"{os.path.getsize(out)/1e6:.1f} MB")
    return out

def ensure_bnwiki():
    # Reuse any attached/local saved dataset first, otherwise reproduce wikibn.ipynb online.
    for root in ["/kaggle/input", "/kaggle/working", "."]:
        if not os.path.isdir(root):
            continue
        for hit in glob.glob(os.path.join(root, "**", "dataset_info.json"), recursive=True):
            d = os.path.dirname(hit)
            if "bnwiki" in d.lower() or "wikipedia" in d.lower():
                return d
    if not USE_RETRIEVAL:
        return None
    out = os.path.join(RESOURCE_DIR, "bnwiki")
    if os.path.exists(os.path.join(out, "dataset_info.json")):
        return out
    assert ONLINE, "Internet is required to download Bengali Wikipedia"
    from datasets import load_dataset
    print('downloading Bengali Wikipedia:', WIKI_DATASET)
    wiki = load_dataset(WIKI_DATASET[0], WIKI_DATASET[1], split="train")
    print("bnwiki articles:", len(wiki))
    wiki.save_to_disk(out)
    return out

MODEL_PATH  = maybe_snapshot(TIGER_ID, "tigerllm-9b-it", needed=True)
QWEN_PATH   = maybe_snapshot(QWEN_ID,  "qwen3.5-9b", needed=NEED_QWEN)
QA_PATH     = None   # QA verifier intentionally disabled unless you set USE_QA_VERIFY and provide/choose a QA model
CLF_PATH    = maybe_snapshot(CLF_ID, "banglabert", needed=USE_CLF)
SQUAD_TARBALL_PATH = ensure_squad_tarball() if (USE_SQUAD_REFS or USE_CTX_MATCH or USE_CLF) else None
BNWIKI_PATH = ensure_bnwiki() if USE_RETRIEVAL else None

print("tigerllm:", MODEL_PATH)
print("qwen    :", QWEN_PATH)
print("encoder :", CLF_PATH)
print("squad   :", SQUAD_TARBALL_PATH)
print("bnwiki  :", BNWIKI_PATH)

if USE_QA_VERIFY and not QA_PATH:
    USE_QA_VERIFY = False; print("QA model missing -> graft 2 disabled")
if USE_RETRIEVAL and not BNWIKI_PATH:
    USE_RETRIEVAL = False; print("bnwiki unavailable -> retrieval disabled")
if USE_CLF and not CLF_PATH:
    USE_CLF = False; print("encoder unavailable -> classifier member disabled")
if NEED_QWEN and not QWEN_PATH:
    USE_ENSEMBLE = USE_XLING_EN = USE_XLING_COT = False
    NEED_QWEN = False
    print("qwen unavailable -> Qwen-dependent passes disabled")


def maybe_hf_login():
    """Use HF_TOKEN when available. Some grammar datasets may require it; public datasets work without it."""
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
            os.environ["HF_TOKEN"] = token
        except Exception:
            token = None
    if token:
        try:
            from huggingface_hub import login
            login(token=token, add_to_git_credential=False)
            print("HF login: token available")
        except Exception as e:
            print("HF login skipped:", e)
    else:
        print("HF login: no token found; using public/anonymous access")
    return token

def ensure_grammar_refs():
    """Reproduce grammar.ipynb: download grammar datasets and save csv/parquet/jsonl outputs."""
    if not USE_GRAMMAR:
        return None
    base = os.path.join(WORK_DIR, "bangla_grammar_datasets")
    expected = [os.path.join(base, name, f"{name}.parquet") for name in GRAMMAR_REPOS]
    if all(os.path.exists(x) for x in expected):
        print("grammar refs already present:", base)
        return base
    if not ONLINE:
        print("grammar refs unavailable: no internet")
        return None
    maybe_hf_login()
    from datasets import load_dataset
    os.makedirs(base, exist_ok=True)
    for name, repo_id in GRAMMAR_REPOS.items():
        out_dir = os.path.join(base, name)
        os.makedirs(out_dir, exist_ok=True)
        try:
            ds = load_dataset(repo_id)
            train = ds["train"] if "train" in ds else next(iter(ds.values()))
            train.to_csv(os.path.join(out_dir, f"{name}.csv"), index=False)
            train.to_parquet(os.path.join(out_dir, f"{name}.parquet"))
            try:
                train.to_json(os.path.join(out_dir, f"{name}.jsonl"), orient="records", lines=True, force_ascii=False)
            except TypeError:
                train.to_json(os.path.join(out_dir, f"{name}.jsonl"), orient="records", lines=True)
            print("grammar", name, "saved", train.num_rows, "rows ->", out_dir)
        except Exception as e:
            print("grammar", repo_id, "load failed:", e)
    return base

def ensure_kaggle_grammar_refs():
    """Download Kaggle idiom datasets into working so the grammar scanner can use them."""
    if not (USE_GRAMMAR and USE_KAGGLE_BAGDHARA and KAGGLE_GRAMMAR_DATASETS):
        return []
    base = os.path.join(WORK_DIR, "kaggle_grammar_refs")
    made = []
    os.makedirs(base, exist_ok=True)
    if not ONLINE:
        print("Kaggle grammar refs unavailable: no internet")
        return made
    try:
        import kagglehub
    except Exception as e:
        print("kagglehub unavailable; Kaggle Bagdhara skipped:", e)
        return made
    for name, slug in KAGGLE_GRAMMAR_DATASETS.items():
        dest = os.path.join(base, name)
        if os.path.isdir(dest) and any(glob.glob(os.path.join(dest, "**", "*"), recursive=True)):
            print("Kaggle grammar", name, "already present:", dest)
            made.append(dest)
            continue
        try:
            raw = kagglehub.dataset_download(slug)
            if os.path.isdir(dest):
                shutil.rmtree(dest)
            shutil.copytree(raw, dest)
            print("Kaggle grammar", slug, "->", dest)
            made.append(dest)
        except Exception as e:
            print("Kaggle grammar", slug, "download failed:", e)
    return made

def _dataset_to_frame(ds):
    frames = []
    for split in ds.keys():
        try:
            frames.append(ds[split].to_pandas())
        except Exception as e:
            print("  split", split, "to_pandas failed:", e)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _flatten_qa_json(obj):
    """v6.3 fix: SQuAD / BanglaRQA style nested json -> flat QA rows the
    reference mapper understands (question / answers / context / is_answerable)."""
    if not (isinstance(obj, dict) and isinstance(obj.get("data"), list)):
        return None
    rows = []
    for art in obj["data"]:
        if not isinstance(art, dict):
            continue
        for para in (art.get("paragraphs") or [art]):
            if not isinstance(para, dict):
                continue
            ctx = str(para.get("context", ""))
            for qa in para.get("qas", []) or []:
                if not isinstance(qa, dict):
                    continue
                q = qa.get("question_text") or qa.get("question") or ""
                ans, texts = qa.get("answers"), []
                if isinstance(ans, dict):
                    texts = [t for t in (ans.get("answer_text") or ans.get("text") or []) if t]
                elif isinstance(ans, list):
                    for a in ans:
                        t = (a.get("answer_text") or a.get("text")) if isinstance(a, dict) else a
                        if t:
                            texts.append(t)
                rows.append({"question": str(q),
                             "answers": json.dumps([str(t) for t in texts], ensure_ascii=False),
                             "context": ctx,
                             "is_answerable": str(qa.get("is_answerable", ""))})
    return pd.DataFrame(rows) if rows else None

def _save_hf_dataset(repo_id, out_path):
    try:
        from datasets import load_dataset
        ds = load_dataset(repo_id)
        df = _dataset_to_frame(ds)
    except Exception as e:
        # v6.3 fix: metadata-version mismatches (e.g. SplitInfo 'description')
        # break load_dataset on older repos -> read the raw data files directly
        print(f"{repo_id}: load_dataset failed ({e}) -> raw-file fallback")
        from huggingface_hub import snapshot_download
        raw = snapshot_download(repo_id, repo_type="dataset")
        frames = []
        for p in sorted(glob.glob(os.path.join(raw, "**", "*"), recursive=True)):
            base = os.path.basename(p).lower()
            try:
                if p.endswith(".parquet"):
                    frames.append(pd.read_parquet(p))
                elif p.endswith(".csv"):
                    frames.append(pd.read_csv(p))
                elif p.endswith(".tsv"):
                    frames.append(pd.read_csv(p, sep="\t"))
                elif p.endswith(".jsonl"):
                    frames.append(pd.read_json(p, lines=True))
                elif p.endswith(".json") and "dataset_info" not in base:
                    with open(p, encoding="utf-8") as f:
                        obj = json.load(f)
                    flat = _flatten_qa_json(obj)
                    frames.append(flat if flat is not None
                                  else pd.DataFrame(obj if isinstance(obj, list) else [obj]))
            except Exception:
                pass
        if not frames:
            df = pd.DataFrame()
        else:
            try:
                df = pd.concat(frames, ignore_index=True)
            except Exception:
                df = max(frames, key=len)
        # parquet-safe: stringify non-scalar object values
        for c in df.columns:
            if df[c].dtype == object:
                df[c] = df[c].map(lambda v: v if isinstance(v, str) or pd.isna(v)
                                  else json.dumps(v, ensure_ascii=False, default=str))
    if df.empty:
        raise ValueError("empty dataset")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_parquet(out_path)
    print("saved", repo_id, df.shape, "->", out_path)
    return out_path

def _hf_search_ids(queries, must=(), limit=12):
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        seen = []
        for q in queries:
            for item in api.list_datasets(search=q, limit=limit):
                rid = getattr(item, "id", None) or getattr(item, "repo_id", None)
                if not rid or rid in seen:
                    continue
                low = rid.lower()
                if all(m.lower() in low for m in must):
                    seen.append(rid)
        return seen
    except Exception as e:
        print("HF dataset search failed:", e)
        return []

def ensure_extra_qa_refs():
    """Download recommended Bengali QA references for exact prompt-answer matching."""
    if not USE_EXTRA_QA_REFS:
        return []
    base = os.path.join(WORK_DIR, "extra_qa_refs")
    os.makedirs(base, exist_ok=True)
    files = []
    if not ONLINE:
        print("extra QA refs unavailable: no internet")
        return files
    maybe_hf_login()
    repo_map = dict(EXTRA_QA_REPOS)
    for name, spec in EXTRA_QA_SEARCH.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        hits = _hf_search_ids(spec.get("queries", []), spec.get("must", []))
        if hits:
            repo_map[name] = hits[0]
            print("HF search", name, "->", hits[0])
        else:
            print("HF search", name, "found no loadable candidate")
    for name, repo_id in repo_map.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        try:
            files.append(_save_hf_dataset(repo_id, out))
        except Exception as e:
            print("extra QA", repo_id, "load failed:", e)
    return files

def ensure_optional_cb_refs():
    """Download optional closed-book culture/linguistic benchmarks such as BLUCK."""
    if not USE_CB_REFS:
        return []
    base = os.path.join(WORK_DIR, "cb_refs_optional")
    os.makedirs(base, exist_ok=True)
    files = []
    if not ONLINE:
        print("optional cb refs unavailable: no internet")
        return files
    maybe_hf_login()
    for name, spec in CB_REF_SEARCH.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        hits = _hf_search_ids(spec.get("queries", []), spec.get("must", []))
        if not hits:
            print("optional cb", name, "not found on HF search")
            continue
        try:
            print("optional cb", name, "->", hits[0])
            files.append(_save_hf_dataset(hits[0], out))
        except Exception as e:
            print("optional cb", hits[0], "load failed:", e)
    return files

def ensure_cb_refs():
    """Reproduce cb-refs.ipynb: download BCS and Bangla-Math references to parquet."""
    if not USE_CB_REFS:
        return None
    base = os.path.join(WORK_DIR, "cb_refs")
    expected = [os.path.join(base, f"{stem}.parquet") for stem in CB_REF_REPOS]
    if all(os.path.exists(x) for x in expected):
        print("closed-book refs already present:", base)
        return base
    if not ONLINE:
        print("closed-book refs unavailable: no internet")
        return None
    maybe_hf_login()
    from datasets import load_dataset
    os.makedirs(base, exist_ok=True)
    for stem, repo in CB_REF_REPOS.items():
        try:
            ds = load_dataset(repo)
            df = pd.concat([ds[split].to_pandas() for split in ds.keys()], ignore_index=True)
            out = os.path.join(base, f"{stem}.parquet")
            df.to_parquet(out)
            print("cb-ref", stem, df.shape, "| columns:", list(df.columns))
        except Exception as e:
            print(repo, "load_dataset failed:", e, "-> raw snapshot fallback")
            try:
                from huggingface_hub import snapshot_download
                snapshot_download(repo, repo_type="dataset",
                                  local_dir=os.path.join(base, f"{stem}_raw"),
                                  local_dir_use_symlinks=False, resume_download=True)
            except Exception as e2:
                print(repo, "raw snapshot fallback failed:", e2)
    return base

# Build optional references before scanning for files. These are no longer attachments.
GRAMMAR_BASE_DIR = ensure_grammar_refs() if USE_GRAMMAR else None
KAGGLE_GRAMMAR_DIRS = ensure_kaggle_grammar_refs() if USE_GRAMMAR else []
CB_REF_BASE_DIR = ensure_cb_refs() if USE_CB_REFS else None
OPTIONAL_CB_FILES = ensure_optional_cb_refs() if USE_CB_REFS else []
EXTRA_QA_FILES = ensure_extra_qa_refs() if USE_EXTRA_QA_REFS else []

# ---- v6.5: knowledge datasets (manually-uploaded local CSVs) -----------------
# NCTB textbook QA (passage+Q+A) and a Bangla knowledge MCQ set. Measured exact
# test-overlap is small (NCTB 0, bqad ~8 rows), so these help mainly as: closed-book
# reference coverage, REAL classifier pairs in the C1/C2 cultural domain (MCQ
# distractors are human-written plausible wrong answers = natural hallucinations),
# retrieval passages, and Phase-2 generalization. Attach the CSVs as a Kaggle dataset;
# each piece disables itself if its file is absent.
USE_KNOWLEDGE_DS = True
USE_BCS = True          # BCS answer-key MCQs (Bangla language & literature)
USE_SATT = True         # v6.13: SATT GK MCQ bank (satt_gk_mcq.csv, attached as a Kaggle dataset)
# v6.11: MCQ pairs (bqad + BCS) as CLASSIFIER training data measurably hurt it. Two runs:
#   KD=11,263 pairs -> labeled AUC all .822 | grounded .971 | closed .584
#   KD=19,129 pairs -> labeled AUC all .803 | grounded .954 | closed .558  (p_clf closed coef -0.13)
# More closed-book MCQ noise degraded BOTH tracks and crowded the same-shape synthetic
# squad pairs out of the cap (15,562 -> 7,696). Their REFERENCES and exact-match TIERS are
# valuable and stay on; only the classifier pairs are dropped. Set True to A/B.
KD_CLF_MCQ = False
BCS_URL = ("https://huggingface.co/datasets/azminetoushikwasi/bangla-bcs-qs/"
           "resolve/main/cleaned_bangla-data_20240901_001128.json")
KD_CLF_ROWS, KD_PASSAGE_FILE, BCS_RAW = [], None, []
SATT_RAW = []   # v6.13: (question, gold_option_TEXT, [distractor_TEXTs])

_ALLNONE = re.compile(r"উপরের\s*সব|সবক[টয়]ি|সবগুলো|সব[গক]ুলি|কোনটিই\s*ন[য়া]|কোনোটিই\s*ন[য়া]|উপরের\s*কোনটিই|all\s+of\s+the\s+above|none\s+of\s+the\s+above", re.I)

def _samestr(a, b):
    return re.sub(r"\s+", " ", str(a)).strip().casefold() == re.sub(r"\s+", " ", str(b)).strip().casefold()

def ensure_knowledge_datasets():
    global KD_PASSAGE_FILE
    if not USE_KNOWLEDGE_DS:
        return []
    ref_files, passages = [], []
    # -- Bangla knowledge MCQ: question + correct option (+ distractors as negatives) --
    bq = find_file("bqad2025.csv")
    if bq:
        try:
            d = pd.read_csv(bq)
            rr, npair = [], 0
            for _, r in d.iterrows():
                q = str(r.get("Question", "")).strip()
                L = str(r.get("Answer", "")).strip().upper()
                gold = str(r.get(L, "")).strip() if L in ("A", "B", "C", "D") else ""
                if not (q and gold and gold.lower() != "nan"):
                    continue
                rr.append({"question": q, "answers": gold})
                if KD_CLF_MCQ:
                    KD_CLF_ROWS.append(("", q, gold, 1)); npair += 1
                    # ONE distractor per MCQ keeps the classifier pool balanced (a full
                    # A/B/C/D expansion is 3:1 hallucinated and biases the member)
                    for opt_l in ("A", "B", "C", "D"):
                        opt = str(r.get(opt_l, "")).strip()
                        if opt and opt.lower() != "nan" and not _samestr(opt, gold):
                            KD_CLF_ROWS.append(("", q, opt, 0)); npair += 1
                            break
            out = os.path.join(RESOURCE_DIR, "bqad_refs.parquet")
            pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
            print(f"knowledge MCQ (bqad): {len(rr)} refs + {npair} classifier pairs")
        except Exception as e:
            print("bqad2025 load failed:", e)
    # -- NCTB textbook QA (local): passage + question + answer --
    nc = find_file("Textbook Dataset from NCTB.csv", "nctb.csv", "nctb_textbook.csv")
    if nc:
        try:
            d = pd.read_csv(nc)
            qcol = next((c for c in d.columns if re.search(r"question", str(c), re.I)), None)
            acol = next((c for c in d.columns if re.search(r"ans", str(c), re.I)), None)
            # v6.9: 'AnsText' also matches the passage regex via `text`; exclude the
            # already-claimed columns so column ORDER can never decide the mapping
            pcol = next((c for c in d.columns
                         if c not in (qcol, acol)
                         and re.search(r"passage|context|evidence|text", str(c), re.I)), None)
            rr = []
            if qcol and acol:
                for _, r in d.iterrows():
                    q, a = str(r[qcol]).strip(), str(r[acol]).strip()
                    p = str(r[pcol]).strip() if pcol else ""
                    if q and a and a.lower() != "nan":
                        rr.append({"question": q, "answers": a})
                        KD_CLF_ROWS.append((p[:1200] if (p and p.lower() != "nan") else "", q, a, 1))
                    if p and p.lower() != "nan":
                        passages.append((q[:60], p))
                out = os.path.join(RESOURCE_DIR, "nctb_local_refs.parquet")
                pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
                print(f"NCTB local: {len(rr)} QA refs + {len(passages)} passages")
        except Exception as e:
            print("NCTB local load failed:", e)
    # -- BCS answer-key MCQs: `answer` is the 1-BASED OPTION INDEX, not the text.
    #    Mapping it naively (the old cb_refs path) put "1".."4" in the reference pool,
    #    which is why this source was junk. Resolve index -> option text here.
    if USE_BCS:
        bp = find_file("cleaned_bangla-data_20240901_001128.json", "bangla-bcs-qs.json")
        if not bp and ONLINE:
            try:
                import urllib.request
                # name must NOT match find_cb_files() - the raw file has an `options` column
                bp = os.path.join(RESOURCE_DIR, "bcs_answerkey_raw.json")
                if not os.path.exists(bp):
                    urllib.request.urlretrieve(BCS_URL, bp)
            except Exception as e:
                print("BCS download failed:", e); bp = None
        if bp:
            try:
                with open(bp, encoding="utf-8") as f:
                    recs = json.load(f)
                rr, bad = [], 0
                for r in recs:
                    q = str(r.get("question", "")).strip()
                    opts = [str(o).strip() for o in (r.get("options") or []) if str(o).strip()]
                    try:
                        idx = int(str(r.get("answer", "")).strip()) - 1
                    except Exception:
                        bad += 1; continue
                    if not (q and opts and 0 <= idx < len(opts)):
                        bad += 1; continue
                    gold = opts[idx]
                    dist = [o for j, o in enumerate(opts) if j != idx and not _samestr(o, gold)]
                    BCS_RAW.append((q, gold, dist))
                    rr.append({"question": q, "answers": gold})
                    if KD_CLF_MCQ:
                        KD_CLF_ROWS.append(("", q, gold, 1))
                        if dist:
                            KD_CLF_ROWS.append(("", q, dist[0], 0))   # 1 distractor keeps balance
                out = os.path.join(RESOURCE_DIR, "bcs_refs.parquet")
                pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
                print(f"BCS answer-key: {len(BCS_RAW)} questions resolved "
                      f"(index->option text), {bad} unparseable")
            except Exception as e:
                print("BCS load failed:", e)

    # -- v6.13: SATT GK MCQ bank. `Answer` is the option LETTER (A/B/C/D), NOT the
    #    answer text and NOT a 1-based index. Resolve letter -> option TEXT here, the
    #    same discipline Layer 1g applies to the BCS bank (whose `answer` is an index;
    #    mapping it naively once put "1".."4" into the reference pool). Distractors are
    #    the OTHER options' TEXT. Classifier pairs are deliberately NOT built from these
    #    (v6.11 measured MCQ distractor pairs as harmful: closed AUC .584 -> .558).
    for _bank_name, _bank_file in (("SATT GK", "satt_gk_mcq.csv"), ("BCS GK", "bcs_gk_mcq.csv"),
                                   ("BCS Bangla", "bcs_bangla_mcq.csv"),
                                   ("BCS Geography", "bcs_geo_mcq.csv"),
                                   ("BCS Science", "bcs_science_mcq.csv")):
        sp = find_file(_bank_file) if USE_SATT else None
        if not sp:
            print(f"{_bank_name}: {_bank_file} not found under /kaggle/input - skipped")
        else:
            try:
                d = pd.read_csv(sp)
                _n0 = len(SATT_RAW)
                need = {"Question", "A", "B", "C", "D", "Answer"}
                missing = need - set(d.columns)
                if missing:
                    raise ValueError(f"missing columns {sorted(missing)}")
                rr, bad = [], 0
                for _, r in d.iterrows():
                    q = str(r.get("Question", "")).strip()
                    L = str(r.get("Answer", "")).strip().upper()
                    if L not in ("A", "B", "C", "D"):
                        bad += 1; continue
                    gold = str(r.get(L, "")).strip()
                    if not (q and gold and gold.lower() != "nan"):
                        bad += 1; continue
                    opts = [str(r.get(c, "")).strip() for c in ("A", "B", "C", "D")]
                    opts = [o for o in opts if o and o.lower() != "nan"]
                    if len(opts) < 2:
                        bad += 1; continue
                    # v6.13 guard: a gold of "all/none of the above" breaks BOTH tiers.
                    # If gold == "সবগুলো"/"all of the above", every OTHER option is TRUE,
                    # so distractor->0 would manufacture false hallucinations (observed on
                    # the Bangla bank: "বাগযন্ত্রের অংশ কোনটি?" gold "উপরের সবকটি" -> the
                    # true answer স্বরযন্ত্র gets called a distractor). If gold ==
                    # "কোনটিই নয়", the gold TEXT is not an answer and is junk in GOLD.
                    # Either way the row is unusable -> drop it.
                    if _ALLNONE.search(gold):
                        bad += 1; continue
                    dist = [o for o in opts if not _samestr(o, gold)]
                    SATT_RAW.append((q, gold, dist))
                    rr.append({"question": q, "answers": gold})
                out = os.path.join(RESOURCE_DIR, f"{_bank_file.replace('.csv','')}_refs.parquet")
                pd.DataFrame(rr).to_parquet(out); ref_files.append(out)
                _added = len(SATT_RAW) - _n0
                print(f"{_bank_name}: {_added} questions resolved (letter->option TEXT), {bad} skipped"
                      + (f" | e.g. gold={SATT_RAW[_n0][1]!r} dist={SATT_RAW[_n0][2][:2]}" if _added else ""))
            except Exception as e:
                print(f"{_bank_name} load failed:", e)

    if passages:
        KD_PASSAGE_FILE = os.path.join(RESOURCE_DIR, "nctb_passages.parquet")
        pd.DataFrame(passages, columns=["title", "text"]).drop_duplicates("text").to_parquet(KD_PASSAGE_FILE)
    return ref_files

KD_REF_FILES = ensure_knowledge_datasets() if USE_KNOWLEDGE_DS else []
# v6.13: hand SATT's (question, gold, distractors) to Layer 1g - it already builds
# exact gold/distractor pairs, quoted-term tiers, drops answer-key conflicts across
# exams, and reports labeled precision. No parallel plumbing needed.
if SATT_RAW:
    BCS_RAW.extend(SATT_RAW)
    print(f"Layer 1g pool: +{len(SATT_RAW)} SATT questions (BCS_RAW now {len(BCS_RAW)})")
EXTRA_QA_FILES = list(EXTRA_QA_FILES) + list(KD_REF_FILES)

def find_grammar_files():
    out, pat = [], re.compile(GRAMMAR_PATH_RE, re.I)
    for root in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            if "competitions" in dirpath:
                continue
            for fn in files:
                if fn.lower().endswith((".csv", ".tsv", ".json", ".jsonl", ".parquet")):
                    full = os.path.join(dirpath, fn)
                    if pat.search(full):
                        out.append(full)
    return out

GRAMMAR_FILES = find_grammar_files() if USE_GRAMMAR else []

def merge_itemized_json_dirs(files, min_files=50):
    """v6.3 fix: Kaggle idiom corpora ship as thousands of one-record .json files
    (e.g. sakhadib/bagdhara-bangla-idioms-dataset). pd.read_json chokes on scalar
    dicts and the per-file loop would spam/skip - merge each such directory into
    one parquet and hand that to the grammar layer instead."""
    from collections import defaultdict
    by_dir = defaultdict(list)
    for p in files:
        if p.lower().endswith(".json"):
            by_dir[os.path.dirname(p)].append(p)
    big_dirs = {d for d, fs in by_dir.items() if len(fs) >= min_files}
    out_files = [p for p in files if os.path.dirname(p) not in big_dirs]
    merged_dir = os.path.join(WORK_DIR, "merged_grammar")
    for d in sorted(big_dirs):
        os.makedirs(merged_dir, exist_ok=True)
        out = os.path.join(merged_dir,
                           re.sub(r"\W+", "_", os.path.basename(d)) + "_grammar_merged.parquet")
        if os.path.exists(out):
            print("merged grammar file already present:", os.path.basename(out))
        else:
            rows, bad = [], 0
            for p in by_dir[d]:
                try:
                    with open(p, encoding="utf-8") as f:
                        obj = json.load(f)
                    for it in (obj if isinstance(obj, list) else [obj]):
                        if isinstance(it, dict):
                            rows.append({k: (v if isinstance(v, (str, int, float, bool)) or v is None
                                             else json.dumps(v, ensure_ascii=False))
                                         for k, v in it.items()})
                except Exception:
                    bad += 1
            if not rows:
                print(f"merge {d}: no readable records ({bad} bad files)")
                continue
            pd.DataFrame(rows).to_parquet(out)
            print(f"merged {len(rows)} records from {len(by_dir[d])} json files "
                  f"({bad} unreadable) -> {os.path.basename(out)}")
        out_files.append(out)
    return sorted(set(out_files))

if USE_GRAMMAR and GRAMMAR_FILES:
    GRAMMAR_FILES = merge_itemized_json_dirs(GRAMMAR_FILES)
print(f"grammar files: {len(GRAMMAR_FILES)}"
      + (" | e.g. " + ", ".join(os.path.basename(p) for p in GRAMMAR_FILES[:5])
         if GRAMMAR_FILES else " (none)"))
if USE_GRAMMAR and not GRAMMAR_FILES:
    USE_GRAMMAR = False
    print("-> grammar layer disabled (no local/public grammar file found)")

def find_cb_files():
    out, pat = [], re.compile(r"cb_refs|bcs[-_ ]?qs|bcs[-_ ]?question|bangla[-_ ]?math", re.I)
    for root in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, _, files in os.walk(root):
            if "competitions" in dirpath:
                continue
            for fn in files:
                if fn.lower().endswith((".csv", ".json", ".jsonl", ".parquet")):
                    full = os.path.join(dirpath, fn)
                    if pat.search(full):
                        out.append(full)
    return out

CB_FILES = find_cb_files() if USE_CB_REFS else []
CB_FILES = sorted(set(CB_FILES + list(globals().get("OPTIONAL_CB_FILES", []))))
print("cb-ref files:", CB_FILES or "none")
if USE_CB_REFS and not CB_FILES:
    USE_CB_REFS = False
    print("-> closed-book reference layer disabled (no local/public cb-ref file found)")


# ---- v6.3: Phase-2 source-base expansion + starter-notebook resources ---------
USE_NEWS_CORPUS = True    # XL-Sum Bengali (BBC Bangla) -> retrieval index (newspapers domain)
USE_RETR_SEARCH = True    # runtime HF search: Banglapedia / bdlaws / NCTB corpora (fail-soft)
RETRIEVAL_SEARCH = {
    "banglapedia": {"queries": ["banglapedia"], "must": ["banglapedia"]},
    # v6.12: known repo id first (47,429 sections of bdlaws.minlaw.gov.bd - an
    # announced Phase-2 source the HF search missed); search stays as fallback.
    "bdlaws":      {"repo": "anisafifi/bd-laws",
                    "queries": ["bdlaws bangladesh law", "bangladesh legislation corpus"], "must": ["bdlaw"]},
    "nctb":        {"queries": ["NCTB textbook bangla", "nctb bengali"], "must": ["nctb"]},
}

def ensure_news_corpus():
    if not (USE_NEWS_CORPUS and USE_RETRIEVAL):
        return None
    out = os.path.join(RESOURCE_DIR, "xlsum_bn.parquet")
    if os.path.exists(out):
        print("news corpus already present:", out)
        return out
    if not ONLINE:
        print("news corpus unavailable: no internet")
        return None
    try:
        # v6.3 fix: xlsum is a script-based repo (unsupported by datasets>=4) ->
        # pull the raw language tarball and parse the jsonl splits directly
        import tarfile, urllib.request
        tb = os.path.join(RESOURCE_DIR, "xlsum_bn.tar.bz2")
        if not os.path.exists(tb):
            urllib.request.urlretrieve(
                "https://huggingface.co/datasets/csebuetnlp/xlsum/resolve/main/"
                "data/bengali_XLSum_v2.0.tar.bz2", tb)
        rows = []
        with tarfile.open(tb, "r:bz2") as tf:
            for name in tf.getnames():
                if not name.endswith(".jsonl"):
                    continue
                with tf.extractfile(name) as fh:
                    for line in fh:
                        try:
                            o = json.loads(line)
                            if o.get("title") and o.get("text"):
                                rows.append((o["title"], o["text"]))
                        except Exception:
                            pass
        df_n = pd.DataFrame(rows, columns=["title", "text"])
        df_n.to_parquet(out)
        print(f"news corpus: {len(df_n)} BBC Bangla articles -> {out}")
        return out
    except Exception as e:
        print("news corpus (xlsum) failed:", e)
        return None

def ensure_retrieval_extras():
    if not (USE_RETR_SEARCH and USE_RETRIEVAL):
        return []
    base = os.path.join(RESOURCE_DIR, "retrieval_extras")
    os.makedirs(base, exist_ok=True)
    files = []
    if not ONLINE:
        return files
    for name, spec in RETRIEVAL_SEARCH.items():
        out = os.path.join(base, f"{name}.parquet")
        if os.path.exists(out):
            files.append(out); continue
        hits = ([spec["repo"]] if spec.get("repo") else []) \
               + _hf_search_ids(spec["queries"], spec["must"])
        if not hits:
            print("retrieval extra", name, "not found on HF search")
            continue
        for hid in hits:
            try:
                print("retrieval extra", name, "->", hid)
                files.append(_save_hf_dataset(hid, out))
                break
            except Exception as e:
                print("retrieval extra", hid, "failed:", e)
    return files

NEWS_CORPUS_PATH      = ensure_news_corpus()
RETRIEVAL_EXTRA_FILES = ensure_retrieval_extras()
# v6.5: NCTB textbook passages join the retrieval corpus (defined above; appended
# here because RETRIEVAL_EXTRA_FILES only exists from this point on)
if KD_PASSAGE_FILE and USE_RETRIEVAL:
    RETRIEVAL_EXTRA_FILES = list(RETRIEVAL_EXTRA_FILES) + [KD_PASSAGE_FILE]
    print(f"retrieval corpus += NCTB passages ({os.path.basename(KD_PASSAGE_FILE)})")

# BenHalluEval files come from the attached `benhallu-data` Kaggle dataset
# (zip the folder and attach it; files are found by name anywhere under /kaggle/input)
BH_QA4000_PATH = find_file("benhallu_qa_4000.csv", "qa_4000.csv")
BH_QADS_PATH   = find_file("benhallu_qa_dataset_2509.csv", "banglahallueval_qa_dataset.csv")
BH_GT1000_PATH = find_file("benhallu_qa_gt_1000.csv", "qa_gt_1000.csv")
print("bh qa_4000 :", BH_QA4000_PATH)
print("bh qa_2509 :", BH_QADS_PATH)
print("bh gt_1000 :", BH_GT1000_PATH, "(optional)")
if USE_BENHALLU and not (BH_QA4000_PATH and BH_QADS_PATH):
    USE_BENHALLU = False
    print("=" * 70)
    print("!! BenHalluEval layer DISABLED - benhallu_qa_4000.csv / "
          "benhallu_qa_dataset_2509.csv NOT found under /kaggle/input.")
    print("!! ~900 test rows (36%) will be judged instead of resolved exactly.")
    print("!! Attach the zipped benhallu-data folder as a Kaggle dataset.")
    print("=" * 70)

labeled : /kaggle/input/competitions/bengali-hallucination/dataset samples.json
test    : /kaggle/input/competitions/bengali-hallucination/test set.csv
sample  : /kaggle/input/competitions/bengali-hallucination/sample submission.csv
downloading squad_bn (~8 MB)...
squad_bn tarball: /kaggle/working/olikbochon_resources/refs/squad_bn.tar.bz2 8.4 MB


downloading Bengali Wikipedia: ('wikimedia/wikipedia', '20231101.bn')


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.bn/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  192MB            

20231101.bn/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

...

dataset_infos.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

bd_laws_all.jsonl: reconstructing file:   0%|          |  0.00B / 82.3MB            

bd_laws_all.jsonl: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

saved anisafifi/bd-laws (47429, 7) -> /kaggle/working/olikbochon_resources/retrieval_extras/bdlaws.parquet
retrieval extra nctb not found on HF search
retrieval corpus += NCTB passages (nctb_passages.parquet)
bh qa_4000 : /kaggle/input/datasets/zarifmahir/dataset2/archive (2)/benhallu-data/benhallu_qa_4000.csv
bh qa_2509 : /kaggle/input/datasets/zarifmahir/dataset2/archive (2)/benhallu-data/benhallu_qa_dataset_2509.csv
bh gt_1000 : /kaggle/input/datasets/zarifmahir/dataset2/archive (2)/benhallu-data/benhallu_qa_gt_1000.csv (optional)


## 2 · Load & clean the data
Same three data quirks as the 0.783 run: `"[NULL]"` vs real JSON null contexts, raw-number responses, and mixed Unicode composition forms (NFC-normalize everything or exact matching silently misses).

In [4]:
NO_CONTEXT_VALUES = {"", "nan", "NaN", "[NULL]", "None"}

def nfc(s):
    return unicodedata.normalize("NFC", str(s))

def clean_context(value):
    if pd.isna(value) or str(value).strip() in NO_CONTEXT_VALUES:
        return ""
    return str(value).strip()

with open(LABELED_PATH, encoding="utf-8") as f:
    labeled = pd.DataFrame(json.load(f))
test   = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)

for df in (labeled, test):
    df["context"]     = df["context"].map(clean_context).map(nfc)
    df["prompt_bn"]   = df["prompt_bn"].astype(str).str.strip().map(nfc)
    df["response_bn"] = df["response_bn"].astype(str).str.strip().map(nfc)
    df["grounded"]    = df["context"] != ""

print(f"labeled: {len(labeled)} rows | grounded {int(labeled.grounded.sum())} / closed-book {int((~labeled.grounded).sum())}")
print(f"test   : {len(test)} rows | grounded {int(test.grounded.sum())} / closed-book {int((~test.grounded).sum())}")
print("labeled label balance:", labeled.label.value_counts().to_dict())


labeled: 299 rows | grounded 130 / closed-book 169
test   : 2516 rows | grounded 1361 / closed-book 1155
labeled label balance: {1: 163, 0: 136}


## 3 · Layer 1 — exact-match label transfer
Unambiguous `(prompt, response)` pairs shared with the labeled set inherit their label with zero risk.

In [5]:
def normalize(s):
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\u2010-\u2015\-\u2013\u2014]+", "-", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.rstrip("-\u2014 ?\u0964").strip()

for df in (labeled, test):
    df["p_norm"] = df.prompt_bn.map(normalize)
    df["r_norm"] = df.response_bn.map(normalize)

pair_labels = (labeled.groupby(["p_norm", "r_norm"]).label
                      .agg(lambda s: int(s.iloc[0]) if s.nunique() == 1 else -1)
                      .to_dict())

hits = pd.Series([pair_labels.get((p, r)) for p, r in zip(test.p_norm, test.r_norm)],
                 index=test.index, dtype="float")
test["lookup_label"] = hits.where(hits.isin([0.0, 1.0]))

n_prompt_match = test.p_norm.isin(set(labeled.p_norm)).sum()
print(f"prompt matches labeled set : {n_prompt_match} rows")
print(f"exact (prompt,response) label transfers: {int(test.lookup_label.notna().sum())} rows")


prompt matches labeled set : 190 rows
exact (prompt,response) label transfers: 11 rows


## 3b · Layer 1b — reference answers from `squad_bn`
Identical to the 0.783 layer, but v6-online downloads the `squad_bn` tarball automatically when it is not already present.

In [6]:
import tarfile, urllib.request

SQUAD_URL = "https://huggingface.co/datasets/csebuetnlp/squad_bn/resolve/main/data/squad_bn.tar.bz2"

def locate_squad_tarball():
    if globals().get("SQUAD_TARBALL_PATH") and os.path.exists(SQUAD_TARBALL_PATH):
        return SQUAD_TARBALL_PATH
    p = find_file("squad_bn.tar.bz2")
    if p:
        return p
    if ONLINE:
        out = os.path.join(RESOURCE_DIR, "refs", "squad_bn.tar.bz2")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        print("downloading squad_bn (~8 MB) from the HF Hub...")
        urllib.request.urlretrieve(SQUAD_URL, out)
        return out
    return None

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def norm_ans(s):
    """Aggressive normalization for answer comparison (digits, case, punctuation)."""
    s = unicodedata.normalize("NFC", str(s)).translate(BN_DIGITS).casefold()
    s = re.sub(r"[‐-―\-–—]+", "-", s)
    s = re.sub(r"[\"'“”‘’()\[\],;:.!।?]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

SUB_RATIO = 0.70   # v6.9: see below
# v6.10 - `gold appears inside the response`. The match audit measured this sub-rule at
# ~2 correct of 7 fires on the test set (truth taken from BenHalluEval pair labels):
#   gold 'মোহর'  <- response 'ডাক নাম ছিল **আকবরী মোহর**'   (hallucinated)
#   gold 'সি'    <- response 'সি++'                          (C++ is not C)
# vs legitimate:  gold 'খাদিজা' <- 'খাদিজা বিনতে খুওয়াইলিদ'
# No length/ratio guard separates them. It fires 0 times on the labeled set, so there is
# no honest way to tune it. Left ON (unchanged LB behaviour); today all 4 known-bad fires
# are corrected downstream because the BenHallu hall-pair tier overrides containment.
# Set False to A/B it - `match_audit.csv` reports exactly which rows it changes.
ALLOW_GOLD_IN_RESPONSE = True

def contains_gold(response, prompt, golds):
    """Echo-safe containment: a gold answer that already appears in the QUESTION text
    is no evidence (e.g. a person's own name inside a "father's name" question).

    v6.9 - the `response inside gold` direction now needs the response to cover most
    of the gold (SUB_RATIO). It exists to absorb Bengali inflection
    ('রশীদা হামিদ' vs 'রশীদা হামিদের', ratio 0.89), NOT to let a fragment match a long
    sentence gold: with NCTB/BanglaRQA answers in the pool (median 41 chars, 62% over
    30), a bare 'বাংলাদেশ' was matching a 47-char gold it is merely a word of.
    Measured on the labeled set: keeps both legitimate fires, blocks the fragments."""
    r, q = norm_ans(response), norm_ans(prompt)
    if not r:
        return False
    for g0 in golds:
        g = norm_ans(g0)
        if len(g) < 2:
            continue
        if g == r:
            return True
        if ALLOW_GOLD_IN_RESPONSE and g in r and g not in q:
            return True
        if r in g and len(r) >= SUB_RATIO * len(g):
            return True
    return False

GOLD = {}
GOLD_SRC = {}          # v6.10: p_norm -> {source names} for the match audit
tarball = locate_squad_tarball() if USE_SQUAD_REFS else None
if tarball:
    with tarfile.open(tarball, "r:bz2") as tf:
        for split in ("train", "validation", "test"):
            with tf.extractfile(f"squad_bn/{split}.json") as fh:
                data = json.load(fh)
            for art in data["data"]:
                for para in art["paragraphs"]:
                    for qa in para["qas"]:
                        answers = [a["text"] for a in qa.get("answers", []) if a.get("text")]
                        if answers:
                            _k = normalize(qa["question"])
                            GOLD.setdefault(_k, set()).update(answers)
                            GOLD_SRC.setdefault(_k, set()).add("squad_bn")
    print(f"squad_bn: {len(GOLD)} unique answerable questions loaded")
else:
    print("squad_bn unavailable (no tarball and no internet) - reference layer disabled")

QA_Q_HINT = re.compile(r"^(question|question_text|query|prompt|problem|proshno)(_|$)|প্রশ্ন", re.I)
QA_A_HINT = re.compile(r"answers?|answer_text|correct|solution|label|output|উত্তর", re.I)
QA_ANSWERABLE_HINT = re.compile(r"answerable|is_answerable|has_answer|unanswerable", re.I)

def _truthy_answerable(v):
    if pd.isna(v) if not isinstance(v, (list, tuple, dict, np.ndarray)) else False:
        return False
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return bool(v)
    return str(v).strip().lower() not in {"0", "false", "no", "না", "nan", "none", "unanswerable"}

def _answer_texts(v):
    out = []
    if v is None:
        return out
    if isinstance(v, float) and pd.isna(v):
        return out
    if isinstance(v, str):
        s0 = v.strip()
        if not s0 or s0.lower() in {"nan", "none", "[]", "{}"}:
            return out
        if s0[:1] in "[{":
            try:
                return _answer_texts(json.loads(s0))
            except Exception:
                try:
                    return _answer_texts(ast.literal_eval(s0))
                except Exception:
                    pass
        return [s0]
    if isinstance(v, dict):
        for key in ("answer_text", "text", "answers", "answer", "value", "label", "correct_answer"):
            if key in v:
                out.extend(_answer_texts(v[key]))
        return out
    if isinstance(v, (list, tuple, set, np.ndarray, pd.Series)):
        for x in list(v):
            out.extend(_answer_texts(x))
        return out
    return [str(v).strip()]

def _load_ref_frame(path):
    try:
        if path.endswith(".parquet"): return pd.read_parquet(path)
        if path.endswith(".csv"):     return pd.read_csv(path)
        if path.endswith(".jsonl"):   return pd.read_json(path, lines=True)
        if path.endswith(".json"):    return pd.read_json(path)
    except Exception as e:
        print("extra QA skip", path, "->", e)
    return None

def add_extra_qa_refs(paths):
    for path in paths or []:
        df_q = _load_ref_frame(path)
        if df_q is None or df_q.empty:
            continue
        q_col = next((c for c in df_q.columns if QA_Q_HINT.search(str(c))), None)
        a_col = next((c for c in df_q.columns if QA_A_HINT.search(str(c)) and c != q_col), None)
        ans_flag = next((c for c in df_q.columns if QA_ANSWERABLE_HINT.search(str(c))), None)
        if q_col is None or a_col is None:
            print(f"{os.path.basename(path)}: could not map QA columns {list(df_q.columns)} - skipped")
            continue
        n0 = len(GOLD); pairs = 0
        for _, row_q in df_q.iterrows():
            if ans_flag is not None and not _truthy_answerable(row_q[ans_flag]):
                continue
            q = str(row_q[q_col]).strip()
            answers = [a for a in _answer_texts(row_q[a_col]) if 0 < len(str(a).strip()) <= 180]
            if q and answers:
                _k = normalize(q)
                GOLD.setdefault(_k, set()).update(map(str, answers))
                GOLD_SRC.setdefault(_k, set()).add(os.path.basename(path))
                pairs += 1
        print(f"{os.path.basename(path)}: q={q_col!r} a={a_col!r} answerable={ans_flag!r}"
              f" -> {pairs} rows, +{len(GOLD) - n0} unique questions")

add_extra_qa_refs(globals().get("EXTRA_QA_FILES", []))
print(f"total exact QA references: {len(GOLD)} unique questions")

LABELED_CORRECT = labeled[labeled.label == 1].groupby("p_norm").response_bn.agg(set).to_dict()
LABELED_WRONG   = labeled[labeled.label == 0].groupby("p_norm").response_bn.agg(set).to_dict()

def make_ref_text(golds, limit=3):
    return " / ".join(sorted(golds, key=len)[:limit])

labeled["ref_text"]    = labeled.p_norm.map(lambda p: make_ref_text(GOLD[p]) if p in GOLD else "")
labeled["ref_contain"] = labeled.apply(
    lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, GOLD[r.p_norm]), axis=1)

test["ref_pool"]    = test.p_norm.map(lambda p: set(GOLD.get(p, set())) | set(LABELED_CORRECT.get(p, set())))
test["ref_text"]    = test.ref_pool.map(lambda g: make_ref_text(g) if g else "")
test["ref_contain"] = test.apply(
    lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, r.ref_pool), axis=1)
test["wrong_match"] = test.apply(
    lambda r: any(norm_ans(r.response_bn) == norm_ans(w) for w in LABELED_WRONG.get(r.p_norm, ())), axis=1)
test.loc[test.ref_contain, "wrong_match"] = False

n_ref = int((test.ref_text != "").sum())
print(f"\ntest rows with a reference answer     : {n_ref} ({n_ref/len(test):.1%} of test)")
print(f"  resolved by containment -> label 1  : {int(test.ref_contain.sum())}")
print(f"  known-wrong match -> label 0        : {int(test.wrong_match.sum())}")
print(f"  residual -> reference-augmented judge: {int(((test.ref_text != '') & ~test.ref_contain & ~test.wrong_match).sum())}")

lm = labeled[labeled.ref_text != ""]
if len(lm):
    print(f"\nlabeled sanity: {len(lm)} rows matched squad_bn | containment-as-label accuracy "
          f"{(lm.ref_contain.astype(int) == lm.label).mean():.3f}")
    p1, p0 = lm[lm.ref_contain], lm[~lm.ref_contain]
    print(f"  containment -> 1 branch: n={len(p1)}, precision {(p1.label == 1).mean():.3f}")
    print(f"  residual rows: n={len(p0)}, {(p0.label == 0).mean():.1%} truly hallucinated")


squad_bn: 71030 unique answerable questions loaded
banglarqa.parquet: q='question' a='answers' answerable='is_answerable' -> 11022 rows, +11006 unique questions
nctb_qa.parquet: q='question' a='answer' answerable=None -> 12329 rows, +12320 unique questions
bqad_refs.parquet: q='question' a='answers' answerable=None -> 4234 rows, +4209 unique questions
nctb_local_refs.parquet: q='question' a='answers' answerable=None -> 2795 rows, +2774 unique questions
bcs_refs.parquet: q='question' a='answers' answerable=None -> 3938 rows, +3733 unique questions
satt_gk_mcq_refs.parquet: q='question' a='answers' answerable=None -> 1942 rows, +1817 unique questions
bcs_gk_mcq_refs.parquet: q='question' a='answers' answerable=None -> 1371 rows, +1253 unique questions
bcs_bangla_mcq_refs.parquet: q='question' a='answers' answerable=None -> 855 rows, +671 unique questions
bcs_geo_mcq_refs.parquet: q='question' a='answers' answerable=None -> 87 rows, +77 unique questions
bcs_science_mcq_refs.parquet: q='qu

## 3b-ii · Layer 1e — BenHalluEval references (v6.2, attached dataset)
The competition's own methodology paper (brief §12) has a public anonymous repository whose
QA track is TyDiQA-bn — and it overlaps Phase 1 heavily: **936 of 2,511 unique test prompts**
carry a verified gold answer here, **550 test (prompt, response) pairs equal a gold answer
exactly** and **351 equal a GPT-generated hallucinated candidate exactly**. On the labeled
sample those two exact-pair signals measured **100% precision (66/66 and 31/31)**. The data ships as the attached `benhallu-data` Kaggle dataset (see its README; the layer disables itself if the attachment is missing). This layer:
1. merges the gold answers into the existing `GOLD` reference pool (containment + ref-judge);
2. builds four exact-pair tiers — gold-pair→1, hallucinated-pair→0, and two weaker tiers from
   real zero-shot LLM answers with judge scores — all **gated in §7** (≥0.95 precision, n≥8);
3. prepares ~13k *real-distribution* training pairs for the classifier member (labeled
   prompts excluded, per the contamination discipline).
⚠️ Phase 2 contains no BenHalluEval samples (organizer disclaimer) — this is a Phase-1 layer.

In [7]:
# ---- v6.2/v6.3: BenHalluEval data from the attached `benhallu-data` dataset ---
# (replaces the embedded payload of earlier revisions - attach the zipped folder
#  as a Kaggle dataset; paths were resolved in the configuration cell)
if USE_BENHALLU:
    bh_hall_raw = pd.read_csv(BH_QA4000_PATH)
    bh_qa = pd.read_csv(BH_QADS_PATH)[
        ["id", "question", "correct_answer", "deepseek_answer", "deepseek_score",
         "gemma_answer", "gemma_score", "qwen_answer", "qwen_score"]]
    bh_ctx = {}
    for sid, c in zip(bh_hall_raw.source_id, bh_hall_raw.context):
        bh_ctx.setdefault(str(sid), str(c) if str(c).lower() != "nan" else "")
    if BH_GT1000_PATH:
        _gt = pd.read_csv(BH_GT1000_PATH)
        for sid, c in zip(_gt.id, _gt.context):
            bh_ctx.setdefault(str(sid), str(c) if str(c).lower() != "nan" else "")
        del _gt
    bh_hall = bh_hall_raw[["source_id", "pattern", "question", "hallucinated_answer"]].copy()
    del bh_hall_raw
    print(f"BenHalluEval loaded: {len(bh_qa)} QA rows | {len(bh_hall)} hallucinated candidates | "
          f"{len(bh_ctx)} contexts")
else:
    print("BenHalluEval data not attached - layer off")


BenHalluEval loaded: 2509 QA rows | 4000 hallucinated candidates | 997 contexts


In [8]:
# ---- v6.2: Layer 1e - BenHalluEval QA references --------------------------------
for df in (labeled, test):
    for c in ("bh_gold", "bh_hall", "bh_m1", "bh_m0"):
        df[c] = False
BH_CLF_ROWS = []

if not USE_BENHALLU:
    print("BenHalluEval layer disabled by flag")
else:
    bh_qa["pn"]   = bh_qa.question.map(normalize)
    bh_hall["pn"] = bh_hall.question.map(normalize)

    def _clean(a):
        a = str(a).strip()
        return a if a and a.lower() != "nan" else ""

    # 1) gold answers extend the existing GOLD reference pool (feeds ref_text /
    #    containment / the ref-judge branch exactly like squad_bn refs)
    n0 = len(GOLD)
    for pn, a in zip(bh_qa.pn, bh_qa.correct_answer):
        a = _clean(a)
        if a:
            GOLD.setdefault(pn, set()).add(a)
            GOLD_SRC.setdefault(pn, set()).add("benhallu")
    print(f"BenHalluEval gold refs: +{len(GOLD) - n0} new questions (GOLD now {len(GOLD)})")

    # 2) exact-pair lookups. On the labeled sample these measured 100% precision
    #    (gold-pair 66/66 -> 1, hall-pair 31/31 -> 0); still gated in section 7.
    BH_GOLD_PAIRS = {(pn, norm_ans(a)) for pn, a in zip(bh_qa.pn, bh_qa.correct_answer) if _clean(a)}
    BH_HALL_PAIRS = {(pn, norm_ans(h)) for pn, h in zip(bh_hall.pn, bh_hall.hallucinated_answer) if _clean(h)}
    _both = BH_GOLD_PAIRS & BH_HALL_PAIRS
    BH_GOLD_PAIRS -= _both
    BH_HALL_PAIRS -= _both
    if _both:
        print(f"dropped {len(_both)} ambiguous pairs present in both pools")

    # 3) real zero-shot LLM answers with judge scores -> weaker, separately gated tiers
    BH_M1_PAIRS, BH_M0_PAIRS = set(), set()
    for m in ("deepseek", "gemma", "qwen"):
        for pn, a, s in zip(bh_qa.pn, bh_qa[f"{m}_answer"], bh_qa[f"{m}_score"]):
            a = _clean(a)
            if not a or pd.isna(s):
                continue
            (BH_M1_PAIRS if int(s) == 1 else BH_M0_PAIRS).add((pn, norm_ans(a)))
    BH_M0_PAIRS -= BH_GOLD_PAIRS | BH_M1_PAIRS   # gold / any judged-correct outranks judged-wrong
    BH_M1_PAIRS -= BH_HALL_PAIRS

    for df in (labeled, test):
        keys = list(zip(df.p_norm, df.response_bn.map(norm_ans)))
        df["bh_gold"] = [k in BH_GOLD_PAIRS for k in keys]
        df["bh_hall"] = [k in BH_HALL_PAIRS for k in keys]
        df["bh_m1"]   = [k in BH_M1_PAIRS and k not in BH_GOLD_PAIRS for k in keys]
        df["bh_m0"]   = [k in BH_M0_PAIRS and k not in BH_HALL_PAIRS for k in keys]

    for name, mask, want in (("gold-pair", labeled.bh_gold, 1), ("hall-pair", labeled.bh_hall, 0),
                             ("model-correct", labeled.bh_m1, 1), ("model-wrong", labeled.bh_m0, 0)):
        n = int(mask.sum())
        if n:
            print(f"labeled {name:14s}: n={n:3d} precision={(labeled.label[mask] == want).mean():.3f}")
    print(f"test fires: gold {int(test.bh_gold.sum())} | hall {int(test.bh_hall.sum())} | "
          f"m1 {int(test.bh_m1.sum())} | m0 {int(test.bh_m0.sum())}")

    # 4) recompute the reference columns now that GOLD grew (same code as layer 1b)
    labeled["ref_text"]    = labeled.p_norm.map(lambda p: make_ref_text(GOLD[p]) if p in GOLD else "")
    labeled["ref_contain"] = labeled.apply(
        lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, GOLD[r.p_norm]), axis=1)
    test["ref_pool"]    = test.p_norm.map(lambda p: set(GOLD.get(p, set())) | set(LABELED_CORRECT.get(p, set())))
    test["ref_text"]    = test.ref_pool.map(lambda g: make_ref_text(g) if g else "")
    test["ref_contain"] = test.apply(
        lambda r: bool(r.ref_text) and contains_gold(r.response_bn, r.prompt_bn, r.ref_pool), axis=1)
    test["wrong_match"] = test.apply(
        lambda r: any(norm_ans(r.response_bn) == norm_ans(w) for w in LABELED_WRONG.get(r.p_norm, ())), axis=1)
    test.loc[test.ref_contain, "wrong_match"] = False

    n_ref = int((test.ref_text != "").sum())
    print(f"\nafter BenHalluEval merge: {n_ref} test rows with references "
          f"| containment -> 1: {int(test.ref_contain.sum())} "
          f"| conflicts vs hall-pair: {int((test.ref_contain & test.bh_hall).sum())} "
          f"(hall-pair wins at override time)")
    lm = labeled[labeled.ref_text != ""]
    if len(lm):
        print(f"labeled sanity: {len(lm)} rows with refs | containment-as-label accuracy "
              f"{(lm.ref_contain.astype(int) == lm.label).mean():.3f}")

    # 5) real training pairs for the classifier member (labeled prompts excluded so
    #    the section-7 gates stay uncontaminated; test-prompt overlap is deliberate -
    #    this is public external data, same as the squad_bn ref layer)
    _labeled_prompts = set(labeled.p_norm)
    _gold_seen = set()
    for r in bh_qa.itertuples():
        if r.pn in _labeled_prompts:
            continue
        c = bh_ctx.get(str(r.id), "")
        g = _clean(r.correct_answer)
        if g and r.pn not in _gold_seen:
            _gold_seen.add(r.pn)
            BH_CLF_ROWS.append((c, str(r.question), g, 1))
        for m in ("deepseek", "gemma", "qwen"):
            a, s = _clean(getattr(r, f"{m}_answer")), getattr(r, f"{m}_score")
            if a and not pd.isna(s):
                BH_CLF_ROWS.append(("", str(r.question), a, int(int(s) == 1)))
    for r in bh_hall.itertuples():
        if r.pn in _labeled_prompts:
            continue
        BH_CLF_ROWS.append((bh_ctx.get(str(r.source_id), ""), str(r.question),
                            _clean(r.hallucinated_answer), 0))
    BH_CLF_ROWS = [x for x in BH_CLF_ROWS if x[2]]
    _n1 = sum(1 for x in BH_CLF_ROWS if x[3] == 1)
    print(f"classifier real pairs: {len(BH_CLF_ROWS)} ({_n1} faithful / {len(BH_CLF_ROWS) - _n1} hallucinated)")


BenHalluEval gold refs: +0 new questions (GOLD now 109378)
labeled gold-pair     : n= 66 precision=1.000
labeled hall-pair     : n= 31 precision=1.000
test fires: gold 550 | hall 351 | m1 1 | m0 2

after BenHalluEval merge: 1294 test rows with references | containment -> 1: 682 | conflicts vs hall-pair: 4 (hall-pair wins at override time)
labeled sanity: 145 rows with refs | containment-as-label accuracy 0.931
classifier real pairs: 13175 (4508 faithful / 8667 hallucinated)


## 3c · Layer 1c — Bangla-grammar references (v4)
বাগধারা / সমাস / এক কথায় প্রকাশ questions answered from your uploaded dataset, through the same
reference mechanism as squad_bn: term matched in the prompt → reference meanings → containment signal +
reference-augmented judge prompt. Column mapping is auto-detected and printed — **check the printout**;
override via `GRAMMAR_TERM_COL` / `GRAMMAR_ANS_COL` in the config cell if it guessed wrong.

In [9]:
# ---- Layer 1c: grammar references (bagdhara / shomash / ek-kothay) ----------
GRAMMAR_Q = re.compile(
    r"বাগধারা|ভাবার্থ|সমাস|ব্যাসবাক্য|এক\s*কথায়|সমার্থক|বিপরীত(?:ার্থক)?\s*শব্দ|সন্ধি|প্রকৃতি\s*ও\s*প্রত্যয়")
QUOTE_RE = re.compile("[\"'\u2018\u201c]([^\"'\u2019\u201d]{1,60})[\"'\u2019\u201d]")

GRAM = {}
if USE_GRAMMAR:
    def _load_any(path):
        try:
            if path.endswith(".csv"):     return pd.read_csv(path)
            if path.endswith(".tsv"):     return pd.read_csv(path, sep="\t")
            if path.endswith(".parquet"): return pd.read_parquet(path)
            if path.endswith(".jsonl"):   return pd.read_json(path, lines=True)
            if path.endswith(".json"):
                try:
                    return pd.read_json(path)
                except ValueError:
                    with open(path, encoding="utf-8") as f:
                        obj = json.load(f)
                    return pd.DataFrame(obj if isinstance(obj, list) else [obj])
        except Exception as e:
            print("skip", path, "->", e)
        return None

    TERM_HINT  = re.compile(r"term|word|idiom|bagdhara|বাগধারা|প্রবাদ|somash|shomash|samas|সমাস|"
                            r"pada|phrase|বাক্যাংশ|proshno|question|প্রশ্ন|title|শিরোনাম|key", re.I)
    ANS_HINT   = re.compile(r"mean|meaning|figurative|answer|artho|ortho|অর্থ|মানে|ভাবার্থ|bekkha|ব্যাখ্যা|"
                            r"bashbakko|ব্যাস|byas|one_word|expres|প্রকাশ|value|solution|explan", re.I)
    # v6.9: `category` was matching bagdhara's idiom-taxonomy column, so values like
    # "cultural"/"religious" were added as REFERENCE ANSWERS - and make_ref_text sorts
    # by length, so the junk led the reference note shown to the judge. Only genuine
    # answer-bearing extras (shomash's samas_type) qualify.
    EXTRA_HINT = re.compile(r"samas_type|samastype|_type$|^type$|ধরন|প্রকার", re.I)
    SKIP_COLS  = re.compile(r"serial|source|url|category|^id$|^index$", re.I)

    _seen_stems = set()
    for path in GRAMMAR_FILES:
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem in _seen_stems:          # csv/jsonl/parquet triplets: load once
            continue
        df_g = _load_any(path)
        if df_g is None or df_g.empty:
            continue
        _seen_stems.add(stem)
        str_cols = [c for c in df_g.columns
                    if df_g[c].dtype == object and not SKIP_COLS.search(str(c))]
        if len(str_cols) < 2:
            continue
        term_col = GRAMMAR_TERM_COL or next((c for c in str_cols if TERM_HINT.search(str(c))), None)
        ans_col  = GRAMMAR_ANS_COL  or next((c for c in str_cols
                                             if ANS_HINT.search(str(c)) and c != term_col), None)
        if term_col is None or ans_col is None:
            lens = {c: df_g[c].astype(str).str.len().mean() for c in str_cols}
            ordered = sorted(lens, key=lens.get)
            term_col = term_col or ordered[0]
            ans_col  = ans_col  or ordered[-1]
        if term_col == ans_col:
            continue
        extra_col = next((c for c in str_cols
                          if EXTRA_HINT.search(str(c)) and c not in (term_col, ans_col)), None)
        n0 = len(GRAM)
        for _, row_g in df_g.iterrows():
            t, a = str(row_g[term_col]).strip(), str(row_g[ans_col]).strip()
            if not (0 < len(t) <= 60 and a and a.lower() != "nan"):
                continue
            refs = {a}
            if extra_col is not None:
                x = str(row_g[extra_col]).strip()
                if x and x.lower() != "nan":
                    refs.add(x)
            for part in re.split(r"\s*/\s*", t):
                part = part.strip()
                if part:
                    GRAM.setdefault(normalize(part), set()).update(refs)
        print(f"{os.path.basename(path)}: term={term_col!r} ans={ans_col!r}"
              f"{' extra=' + repr(extra_col) if extra_col else ''} -> +{len(GRAM) - n0} terms")
    print(f"grammar lookup: {len(GRAM)} terms")

GRAM_KEYS = sorted(GRAM, key=len, reverse=True)

def grammar_refs(prompt):
    """Reference answers for a grammar-style question, else empty set."""
    if not GRAM or not GRAMMAR_Q.search(prompt):
        return set()
    m = QUOTE_RE.search(prompt)
    if m:
        k = normalize(m.group(1))
        if k in GRAM:
            return GRAM[k]
    p_n = normalize(prompt)
    for k in GRAM_KEYS:
        if len(k) >= 3 and k in p_n:
            return GRAM[k]
    return set()

for df in (labeled, test):
    refs = df.prompt_bn.map(grammar_refs)
    df["gram_hit"] = refs.map(bool)
    df["gram_contain"] = [contains_gold(r, p, g) if g else False
                          for r, p, g in zip(df.response_bn, df.prompt_bn, refs)]
    fill = df.gram_hit & (df.ref_text == "")
    df.loc[fill, "ref_text"] = refs[fill].map(make_ref_text)

for name, df in (("labeled", labeled), ("test", test)):
    n_gq = int(df.prompt_bn.map(lambda p: bool(GRAMMAR_Q.search(p))).sum())
    print(f"{name}: grammar-style prompts {n_gq} | with reference {int(df.gram_hit.sum())} "
          f"| containment hits {int(df.gram_contain.sum())}")
if USE_GRAMMAR and labeled.gram_hit.sum():
    gl = labeled[labeled.gram_hit]
    print(f"labeled grammar sanity: containment-as-label accuracy "
          f"{(gl.gram_contain.astype(int) == gl.label).mean():.3f} on n={len(gl)}")

bagdhara.csv: term='bagdhara' ans='meaning' -> +1299 terms
ek_kothay_prokash.csv: term='phrase' ans='one_word' -> +1058 terms
shomash.csv: term='samasta_pada' ans='byasabakya' extra='samas_type' -> +189 terms
grammar lookup: 2546 terms
labeled: grammar-style prompts 14 | with reference 2 | containment hits 0
test: grammar-style prompts 293 | with reference 57 | containment hits 3
labeled grammar sanity: containment-as-label accuracy 1.000 on n=2


## 3c-ii · Layer 1h — field-aware bagdhara idiom tiers (v6.11)

The ~300 quoted-term word-meaning rows (`"X" এর শাব্দিক অর্থ কী?`) are judge-only in this pipeline: `GRAMMAR_Q` deliberately skips শাব্দিক and `USE_KAGGLE_BAGDHARA` is off. The [sakhadib idiom dictionary](https://www.kaggle.com/datasets/sakhadib/bagdhara-bangla-idioms-dataset) (10,361 entries) carries **separate literal and figurative meanings** — and the organizers' hallucination trap on these rows is exactly the literal↔figurative swap.

Field-aware rule: a শাব্দিক/আভিধানিক question is checked against the *literal* field, anything else against the *figurative* field. Match the **asked** field → 1; match only the **other** field (the swap trap) → 0; no signal → row stays with the judges. Measured on labeled closed rows: **12/12** (tier→1 8/8, tier→0 4/4); **116 test verdicts**. Both tiers re-gated at ≥ 0.90 labeled precision in §7.

**Do not attach the Kaggle dataset** — this cell downloads its own copy at runtime (the direct URL works in committed runs; kagglehub does not). Attaching it would also sweep 10k JSONs into the grammar layer and change its 0.839-run behavior; a warning prints if that happens.


In [10]:
# ---- Layer 1h: field-aware bagdhara idiom tiers (v6.11) -----------------------
# Measured 12/12 on labeled closed rows; ~116 test verdicts. Uses only helpers
# defined above (normalize, norm_ans). Self-disables without the dictionary.
USE_IDIOM_TIERS = True
IDM_LIT, IDM_FIG = {}, {}
IDIOM1_OK = IDIOM0_OK = False    # precision-gated in section 7

def _idm_clean(v):
    v = re.sub(r"\s*\([^)]*\)\s*", " ", str(v)).strip()   # strip English parentheticals
    parts = {v} | {p.strip() for p in re.split(r"[;,।]", v) if len(p.strip()) >= 4}
    return {p for p in parts if p}

def _idm_source():
    import glob as _g
    hits = sorted(d for d, _, _ in os.walk("/kaggle/input")
                  if "bagdhara-bangla-idioms" in os.path.basename(d).lower())
    if hits:
        if any("bagdhara_bangla_idioms" in os.path.basename(p) for p in GRAMMAR_FILES):
            print("NOTE: the attached idiom dictionary was ALSO merged into the grammar "
                  "layer (GRAM differs from the 0.839 run). Prefer not attaching it - "
                  "this layer downloads its own copy.")
        return hits[0]
    if ONLINE:
        try:
            import urllib.request, zipfile
            _dst = os.path.join(WORK_DIR, "idm_bagdhara")
            if not os.path.isdir(_dst):
                req = urllib.request.Request(
                    "https://www.kaggle.com/api/v1/datasets/download/"
                    "sakhadib/bagdhara-bangla-idioms-dataset",
                    headers={"User-Agent": "Mozilla/5.0"})
                _zp = os.path.join(WORK_DIR, "idm_bagdhara.zip")
                with urllib.request.urlopen(req, timeout=300) as r, open(_zp, "wb") as f:
                    f.write(r.read())
                with zipfile.ZipFile(_zp) as z:
                    z.extractall(_dst)
            return _dst
        except Exception as e:
            print(f"idiom dictionary download failed ({e})")
    return None

if USE_IDIOM_TIERS:
    import glob as _idm_glob
    _src = _idm_source()
    if _src:
        _n_bad = 0
        for _fp in _idm_glob.glob(os.path.join(_src, "**", "*.json"), recursive=True):
            try:
                with open(_fp, encoding="utf-8") as f:
                    _r = json.load(f)
            except Exception:
                _n_bad += 1
                continue
            if not isinstance(_r, dict) or "idiom" not in _r:
                continue
            _lit = _idm_clean(_r.get("literal_meaning") or "")
            _fig = _idm_clean(_r.get("figurative_meaning_bn") or "")
            for _key in [_r.get("idiom", "")] + list(_r.get("alternative_idioms") or []):
                _key = normalize(_key)
                if _key:
                    IDM_LIT.setdefault(_key, set()).update(_lit)
                    IDM_FIG.setdefault(_key, set()).update(_fig)
        print(f"idiom dictionary: {len(IDM_LIT)} keys loaded"
              + (f" ({_n_bad} unparseable files skipped)" if _n_bad else ""))
    else:
        print("idiom dictionary unavailable - idiom tiers disabled")
else:
    print("idiom tiers disabled by flag")

_IDMQ_RE = re.compile(r'[\"“”‘’\'](.+?)[\"“”‘’\']')
_IDML_RE = re.compile(r"শাব্দিক|আভিধানিক")

def _idm_match(response, golds):
    r_n = norm_ans(response)
    for g0 in golds:
        g = norm_ans(g0)
        if len(g) < 4:
            continue
        if g == r_n or (g in r_n and len(g) >= 6) or (r_n in g and len(r_n) >= 8):
            return True
    return False

def idiom_verdict(prompt, response):
    """1 = response matches the ASKED field's meaning, 0 = it matches only the
    OTHER field (the literal/figurative swap trap), None = no signal."""
    m = _IDMQ_RE.search(str(prompt))
    if not m:
        return None
    t = normalize(m.group(1))
    if t not in IDM_LIT and t not in IDM_FIG:
        return None
    lit, fig = IDM_LIT.get(t, set()), IDM_FIG.get(t, set())
    right, wrong = (lit, fig) if _IDML_RE.search(str(prompt)) else (fig, lit)
    if _idm_match(response, right):
        return 1
    if _idm_match(response, wrong):
        return 0
    return None

for _df in (labeled, test):
    _df["idiom_v"] = np.nan
    if IDM_LIT:
        for _i in _df.index[~_df.grounded]:
            _v = idiom_verdict(_df.at[_i, "prompt_bn"], _df.at[_i, "response_bn"])
            if _v is not None:
                _df.at[_i, "idiom_v"] = _v
if IDM_LIT:
    print(f"idiom verdicts: labeled {int(labeled.idiom_v.notna().sum())} "
          f"(->1 {int((labeled.idiom_v == 1).sum())} / ->0 {int((labeled.idiom_v == 0).sum())}) | "
          f"test {int(test.idiom_v.notna().sum())} "
          f"(->1 {int((test.idiom_v == 1).sum())} / ->0 {int((test.idiom_v == 0).sum())})")


idiom dictionary: 11051 keys loaded
idiom verdicts: labeled 12 (->1 8 / ->0 4) | test 116 (->1 76 / ->0 40)


## 3c · Graft 3 — BM25 retrieval over bn-wiki (closed-book rows)
CPU-only, runs before the judge loads. Inlined inverted-index BM25 (no pip dependency). Retrieved
contexts are stored per row; whether they're *used* is decided in §7 by CV on labeled closed rows.

In [11]:
BN_STOP = set("কী কি কে কোন কত কোথায় কবে কেন হয় হলো ছিল এর ও এবং থেকে জন্য করে একটি টি টির".split())
TOK_RE  = re.compile(r"[\w\u0980-\u09FF]+")

def bm25_tokenize(s):
    s = nfc(s).translate(BN_DIGITS).lower()
    return [t for t in TOK_RE.findall(s) if t not in BN_STOP and len(t) > 1]

class BM25:
    def __init__(self, docs_tokens, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(docs_tokens)
        self.dl = np.array([len(d) for d in docs_tokens], dtype=np.float32)
        self.avgdl = max(float(self.dl.mean()), 1.0)
        postings = {}
        for i, d in enumerate(docs_tokens):
            seen = {}
            for t in d:
                seen[t] = seen.get(t, 0) + 1
            for t, tf in seen.items():
                postings.setdefault(t, ([], []))
                postings[t][0].append(i); postings[t][1].append(tf)
        self.post = {t: (np.array(ids, dtype=np.int32), np.array(tfs, dtype=np.float32))
                     for t, (ids, tfs) in postings.items()}
        self.idf = {t: math.log(1 + (self.N - len(ids) + 0.5) / (len(ids) + 0.5))
                    for t, (ids, _) in self.post.items()}

    def top(self, query_tokens, k=2):
        scores = np.zeros(self.N, dtype=np.float32)
        for t in set(query_tokens):
            if t not in self.post:
                continue
            ids, tfs = self.post[t]
            denom = tfs + self.k1 * (1 - self.b + self.b * self.dl[ids] / self.avgdl)
            scores[ids] += self.idf[t] * tfs * (self.k1 + 1) / denom
        k = min(k, self.N)
        idx = np.argpartition(scores, -k)[-k:]
        idx = idx[np.argsort(-scores[idx])]
        return idx, scores[idx]

labeled["ret_text"] = ""; labeled["ret_conf"] = np.nan
test["ret_text"]    = ""; test["ret_conf"]    = np.nan

if USE_RETRIEVAL:
    import gc
    try:
        from datasets import load_from_disk
        wiki = load_from_disk(BNWIKI_PATH)
        wiki_iter = ({"title": a["title"], "text": a["text"]} for a in wiki)
        n_articles = len(wiki)
    except Exception as e:
        print("load_from_disk failed:", e, "-> pyarrow fallback")
        import pyarrow as pa
        tables = [pa.ipc.open_stream(open(p, "rb")).read_all()
                  for p in sorted(glob.glob(os.path.join(BNWIKI_PATH, "*.arrow")))]
        tbl = pa.concat_tables(tables)
        n_articles = tbl.num_rows
        wiki_iter = ({"title": t, "text": x} for t, x in
                     zip(tbl.column("title").to_pylist(), tbl.column("text").to_pylist()))

    from tqdm.auto import tqdm
    chunks, chunk_tok = [], []
    for art in tqdm(wiki_iter, total=n_articles, desc="chunking bn-wiki"):
        title, lead = str(art["title"]), str(art["text"])[:1300]
        parts = [p.strip() for p in lead.split("\u0964") if p.strip()]
        buf, taken = "", 0
        for p in parts:
            if len(buf) + len(p) > 600 and buf:
                chunks.append(title + " \u2014 " + buf + "\u0964")
                chunk_tok.append(bm25_tokenize(title + " " + buf))
                taken += 1; buf = p
                if taken >= 2:
                    buf = ""; break
            else:
                buf = (buf + "\u0964 " + p) if buf else p
        if buf and taken < 2:
            chunks.append(title + " \u2014 " + buf + "\u0964")
            chunk_tok.append(bm25_tokenize(title + " " + buf))

    # ---- v6.3: Phase-2 source-base expansion (news / searched corpora) ----------
    def _chunk_doc(title, text):
        title, text = str(title).strip(), str(text)[:1300]
        if not re.search(r"[\u0980-\u09FF]", text):
            return   # Bengali-script guard: searched repos may return junk
        parts = [p.strip() for p in text.split("\u0964") if p.strip()]
        buf, taken = "", 0
        for p in parts:
            if len(buf) + len(p) > 600 and buf:
                chunks.append(title + " \u2014 " + buf + "\u0964")
                chunk_tok.append(bm25_tokenize(title + " " + buf))
                taken += 1; buf = p
                if taken >= 2:
                    buf = ""; break
            else:
                buf = (buf + "\u0964 " + p) if buf else p
        if buf and taken < 2:
            chunks.append(title + " \u2014 " + buf + "\u0964")
            chunk_tok.append(bm25_tokenize(title + " " + buf))

    if globals().get("NEWS_CORPUS_PATH"):
        n0 = len(chunks)
        _news = pd.read_parquet(NEWS_CORPUS_PATH)
        for ttl, txt in zip(_news.title.astype(str), _news.text.astype(str)):
            _chunk_doc(ttl[:60], txt)
        del _news
        print(f"retrieval += {len(chunks) - n0} chunks from BBC Bangla news (XL-Sum)")
    for _path in globals().get("RETRIEVAL_EXTRA_FILES", []) or []:
        try:
            df_x = pd.read_parquet(_path)
        except Exception as e:
            print("retrieval extra skip", _path, e); continue
        _scols = [c for c in df_x.columns if df_x[c].dtype == object]
        if not _scols:
            continue
        _lens = {c: df_x[c].astype(str).str.len().mean() for c in _scols}
        _tcol = max(_lens, key=_lens.get)
        _ttlcol = next((c for c in _scols if re.search(r"title|name|heading", str(c), re.I)), None)
        _texts = df_x[_tcol].astype(str).tolist()
        _titles = (df_x[_ttlcol].astype(str).tolist() if _ttlcol
                   else [t[:40] for t in _texts])
        n0, _cap = len(chunks), 120000
        for ttl, txt in zip(_titles, _texts):
            if len(chunks) - n0 >= _cap:
                break
            _chunk_doc(ttl[:60], txt)
        del df_x
        print(f"retrieval += {len(chunks) - n0} chunks from {os.path.basename(_path)}")

    print(len(chunks), "chunks; building BM25 index...")
    bm25 = BM25(chunk_tok)
    del chunk_tok; gc.collect()

    for df, dname in ((labeled, "labeled"), (test, "test")):
        closed_idx = df.index[~df.grounded]
        for i in tqdm(closed_idx, desc=f"retrieve {dname}"):
            q = bm25_tokenize(df.at[i, "prompt_bn"])
            if not q:
                continue
            idx, sc = bm25.top(q, k=2)
            if sc[0] <= 0:
                continue
            df.at[i, "ret_text"] = " ".join(chunks[j] for j in idx)[:1500]
            df.at[i, "ret_conf"] = float(sc[0])
    confs = pd.concat([labeled.ret_conf, test.ret_conf]).dropna().values
    print(f"retrieved for {int((labeled.ret_text != '').sum())} labeled + "
          f"{int((test.ret_text != '').sum())} test closed rows | "
          f"conf p50={np.percentile(confs, 50):.1f} p75={np.percentile(confs, 75):.1f} p90={np.percentile(confs, 90):.1f}")
else:
    print("retrieval disabled")


# ---- v4: context relevance (share of question content-words found in context) ----
def ctx_relevance(prompt, context):
    pt = set(bm25_tokenize(prompt))
    if not pt:
        return 1.0
    ct = set(bm25_tokenize(context))
    return len(pt & ct) / len(pt)

for df in (labeled, test):
    df["ctx_rel"] = [ctx_relevance(p, c) if g else np.nan
                     for p, c, g in zip(df.prompt_bn, df.context, df.grounded)]
_rel = pd.concat([labeled.ctx_rel, test.ctx_rel]).dropna()
print(f"context relevance: p10={_rel.quantile(.10):.2f} p25={_rel.quantile(.25):.2f} "
      f"median={_rel.quantile(.50):.2f} | rows < 0.30: {int((_rel < 0.30).sum())}")


chunking bn-wiki:   0%|          | 0/143069 [00:00<?, ?it/s]

retrieval += 20151 chunks from BBC Bangla news (XL-Sum)
retrieval += 33282 chunks from bdlaws.parquet
retrieval += 534 chunks from nctb_passages.parquet
307445 chunks; building BM25 index...


retrieve labeled:   0%|          | 0/169 [00:00<?, ?it/s]

retrieve test:   0%|          | 0/1155 [00:00<?, ?it/s]

retrieved for 169 labeled + 1154 test closed rows | conf p50=23.2 p75=29.5 p90=37.2
context relevance: p10=0.20 p25=0.33 median=0.50 | rows < 0.30: 261


## 3d · Layer 1b-ctx — paragraph-level squad_bn matching (v5)
Prompt matching covered 69% of grounded rows; this matches the **paragraph** instead, recovering
paraphrased questions. Soft prompt match (token overlap vs the paragraph's own questions) supplies
references; a response matching none of the paragraph's annotated answers is a hallucination signal.
Both tiers are precision-gated in §7 before touching test.

In [12]:
# ---- Layer 1b-ctx: paragraph-level squad_bn matching (v5) -------------------
PARA, PARA_PREFIX = {}, {}
_NUM_RE = re.compile(r"-?\d+(?:\.\d+)?")

def _num(s):
    m = _NUM_RE.search(norm_ans(s).replace(",", ""))
    return float(m.group()) if m else None

if USE_CTX_MATCH and tarball:
    with tarfile.open(tarball, "r:bz2") as tf:
        for split in ("train", "validation", "test"):
            with tf.extractfile(f"squad_bn/{split}.json") as fh:
                data = json.load(fh)
            for art in data["data"]:
                for para in art["paragraphs"]:
                    qas = []
                    for qa in para["qas"]:
                        answers = tuple(a["text"] for a in qa.get("answers", []) if a.get("text"))
                        if answers:
                            qas.append((frozenset(bm25_tokenize(qa["question"])), answers))
                    if not qas:
                        continue
                    k = normalize(para["context"])
                    PARA.setdefault(k, []).extend(qas)
                    PARA_PREFIX.setdefault(k[:200], []).extend(qas)
    print(f"paragraph index: {len(PARA)} unique paragraphs")

def para_qas(context):
    k = normalize(context)
    return PARA.get(k) or PARA_PREFIX.get(k[:200]) or []

def pool_match(response, answers):
    """Generous match (no echo guard): equality/containment either way, or equal numbers."""
    r = norm_ans(response)
    if not r:
        return False
    rn = _num(response)
    for a in answers:
        g = norm_ans(a)
        if len(g) < 2:
            continue
        if g == r or g in r or r in g:
            return True
        an = _num(a)
        if rn is not None and an is not None and abs(rn - an) < 1e-6:
            return True
    return False

from difflib import SequenceMatcher as _SM

def resp_agree(a, b):
    """Do two responses plausibly express the same answer?"""
    x, y = norm_ans(a), norm_ans(b)
    if not x or not y:
        return False
    if x == y or x in y or y in x:
        return True
    nx, ny = _num(a), _num(b)
    if nx is not None and ny is not None:
        return abs(nx - ny) < 1e-6
    return _SM(None, x, y).ratio() >= 0.75

def _tok_match(a, b):
    """Prefix-tolerant token equality - absorbs Bangla inflection
    (কীবোর্ডের ~ কীবোর্ড, আবিষ্কারক ~ আবিষ্কার)."""
    if a == b:
        return True
    return len(a) >= 4 and len(b) >= 4 and a[:4] == b[:4]

def soft_overlap(q_toks, p_toks):
    if not p_toks:
        return 0.0
    hits = sum(1 for p in p_toks if any(_tok_match(p, q) for q in q_toks))
    return hits / len(p_toks)

SOFT_MIN = 0.60
for df in (labeled, test):
    hit, soft, contain, miss = [], [], [], []
    for row in df.itertuples():
        qas = para_qas(row.context) if row.grounded else []
        if not qas:
            hit.append(False); soft.append(False); contain.append(False); miss.append(False)
            continue
        hit.append(True)
        pt = set(bm25_tokenize(row.prompt_bn))
        best_ov, best_ans = 0.0, ()
        for q_toks, answers in qas:
            ov = soft_overlap(q_toks, pt)
            if ov > best_ov:
                best_ov, best_ans = ov, answers
        is_soft = best_ov >= SOFT_MIN
        soft.append(is_soft)
        contain.append(bool(is_soft and contains_gold(row.response_bn, row.prompt_bn, set(best_ans))))
        pool = set()
        for _, answers in qas:
            pool.update(answers)
        miss.append(not pool_match(row.response_bn, pool))
    df["ctx_hit"], df["ctx_soft"] = hit, soft
    df["ctx_contain"], df["ctx_miss"] = contain, miss
    # v5.1: NO ref_text fill - soft-matched refs feed the tier-1 override only,
    # never the prompts (a 0.60 overlap can pick the wrong question from the
    # right paragraph; containment corroboration protects the tier, prompts had none)

for name, df in (("labeled", labeled), ("test", test)):
    g = df[df.grounded]
    print(f"{name}: paragraph-matched {int(g.ctx_hit.sum())}/{len(g)} grounded | "
          f"soft prompt match {int(g.ctx_soft.sum())} | "
          f"containment {int((g.ctx_soft & g.ctx_contain).sum())} | "
          f"pool-miss {int((g.ctx_hit & g.ctx_miss).sum())}")


paragraph index: 21577 unique paragraphs
labeled: paragraph-matched 83/130 grounded | soft prompt match 80 | containment 45 | pool-miss 35
test: paragraph-matched 938/1361 grounded | soft prompt match 937 | containment 546 | pool-miss 370


## 3d-ii · Layer 1f — grounded numeric context-verification (v6.4)
For a row **with context**, a wrong answer is often *checkably* wrong: the response states a
number the passage never mentions (the "digit slightly altered" hallucination the brief
describes). On the labeled grounded rows this measured **precision 1.000 (0 false positives),
recall 28% of grounded hallucinations**. Deterministic, passage-grounded, and independent of
the judge's world knowledge — exactly the faithfulness sub-task a grounded row really is.
The rule abstains when the context contains no digits (numbers spelled as Bengali words) so it
never guesses. A softer token-coverage signal also feeds the meta-stacker.

In [13]:
# ---- Layer 1f: grounded numeric context-verification (v6.4) ------------------
_NUM = re.compile(r"\d+(?:\.\d+)?")
def _digits(s):
    return set(_NUM.findall(nfc(s).translate(BN_DIGITS)))

def ctx_num_contradiction(response, context):
    """Grounded numeric contradiction: response has a number, the context DOES contain
    digits, and some response number is absent from the context -> hallucination (0).
    Returns True (contradiction), False (numbers consistent), or None (abstain:
    no numbers in the response, or the context has no digits to compare against)."""
    rn = _digits(response)
    if not rn:
        return None
    cn = _digits(context)
    if not cn:
        return None
    return not rn.issubset(cn)

CTXCOV_STOP = set("কী কি কে কোন কত কোথায় কবে কেন হয় হলো ছিল এর ও এবং থেকে জন্য করে "
                  "একটি টি টির এই সেই তার অথবা কয়টি".split())
def ctx_coverage(response, context):
    """Share of response content-tokens (prefix-4, inflection-tolerant) found in the
    context. Low coverage weakly signals an unsupported response; used as a stack
    feature only (not override-grade on its own)."""
    rt = [t for t in bm25_tokenize(response) if t not in CTXCOV_STOP and len(t) > 1]
    if not rt:
        return 1.0
    ct = {t[:4] for t in bm25_tokenize(context)}
    return sum(1 for t in rt if t[:4] in ct) / len(rt)

for df in (labeled, test):
    g = df.grounded.values
    df["ctx_num_bad"] = [bool(g[i]) and (ctx_num_contradiction(r, c) is True)
                         for i, (r, c) in enumerate(zip(df.response_bn, df.context))]
    df["ctx_cov"] = [ctx_coverage(r, c) if g[i] else 1.0
                     for i, (r, c) in enumerate(zip(df.response_bn, df.context))]

lg = labeled[labeled.grounded]
fire = lg[lg.ctx_num_bad]
if len(fire):
    print(f"Layer 1f (labeled grounded): numeric-contradiction fires n={len(fire)} "
          f"precision(0)={(fire.label == 0).mean():.3f} | "
          f"recall of grounded halluc {int((fire.label == 0).sum())}/{int((lg.label == 0).sum())}")
print(f"Layer 1f (test grounded): {int(test.ctx_num_bad.sum())} rows flagged as "
      f"numeric contradictions -> candidate label 0 (gated in section 7)")


Layer 1f (labeled grounded): numeric-contradiction fires n=13 precision(0)=1.000 | recall of grounded halluc 13/47
Layer 1f (test grounded): 130 rows flagged as numeric contradictions -> candidate label 0 (gated in section 7)


## 3e · Layer 1d — closed-book external references (v5)
BCS / Bangla-Math question banks as an answer key for the closed-book track. Exact
normalized-prompt matching; the printed match counts **are** the overlap sweep — if they're
near zero, this layer costs nothing and does nothing. Tiers precision-gated in §7.

In [14]:
# ---- Layer 1d: closed-book external references ------------------------------
CB = {}
if USE_CB_REFS and CB_FILES:
    Q_HINT = re.compile(r"question|prompt|proshno|প্রশ্ন|problem|body|text|stem|mcq", re.I)
    A_HINT = re.compile(r"answer|\bans\b|উত্তর|solution|correct|সঠিক|label|output|option|choice", re.I)
    _seen = set()
    for path in CB_FILES:
        stem = os.path.splitext(os.path.basename(path))[0]
        if stem in _seen:
            continue
        df_c = None
        try:
            if path.endswith(".parquet"): df_c = pd.read_parquet(path)
            elif path.endswith(".csv"):   df_c = pd.read_csv(path)
            elif path.endswith(".jsonl"): df_c = pd.read_json(path, lines=True)
            elif path.endswith(".json"):  df_c = pd.read_json(path)
        except Exception as e:
            print("skip", path, "->", e)
            continue
        if df_c is None or df_c.empty:
            continue
        _seen.add(stem)
        str_cols = [c for c in df_c.columns if df_c[c].dtype == object]
        q_col = next((c for c in str_cols if Q_HINT.search(str(c))), None)
        # v6.11: prefer an exact answer column. `options` also matches A_HINT and, in
        # azminetoushikwasi/bangla-bcs-qs, precedes `answer` - which silently mapped the
        # LIST OF CHOICES as the gold answer. That dataset is handled properly by Layer 1g.
        A_STRICT = re.compile(r"^(answer|ans|correct(_answer)?|solution|উত্তর)$", re.I)
        a_col = (next((c for c in df_c.columns if A_STRICT.search(str(c)) and c != q_col), None)
                 or next((c for c in df_c.columns if A_HINT.search(str(c)) and c != q_col), None))
        if q_col is None or a_col is None:
            print(f"{os.path.basename(path)}: could not map columns {list(df_c.columns)} - skipped")
            continue
        _s = df_c[a_col].dropna().head(20)
        if any(isinstance(v, (list, tuple)) or (isinstance(v, str) and v.strip().startswith("["))
               for v in _s):
            print(f"{os.path.basename(path)}: answer column {a_col!r} holds option LISTS - skipped")
            continue
        if "option" in str(a_col).lower():
            print(f"{os.path.basename(path)}: refusing to use {a_col!r} as a gold answer - skipped")
            continue
        n0 = len(CB)
        for q, a in zip(df_c[q_col], df_c[a_col]):
            q, a = str(q).strip(), str(a).strip()
            if q and a and a.lower() != "nan":
                CB.setdefault(normalize(q), set()).add(a)
        print(f"{os.path.basename(path)}: q={q_col!r} a={a_col!r} -> +{len(CB) - n0} questions")
    print(f"closed-book reference lookup: {len(CB)} questions")

for df in (labeled, test):
    cb_hit, cb_contain, cb_miss = [], [], []
    for row in df.itertuples():
        golds = CB.get(row.p_norm) if not row.grounded else None
        if not golds:
            cb_hit.append(False); cb_contain.append(False); cb_miss.append(False)
            continue
        cb_hit.append(True)
        cb_contain.append(bool(contains_gold(row.response_bn, row.prompt_bn, golds)))
        cb_miss.append(not pool_match(row.response_bn, golds))
    df["cb_hit"], df["cb_contain"], df["cb_miss"] = cb_hit, cb_contain, cb_miss
    # v5.1: NO ref_text fill (16 of 20 harmful 1->0 flips traced to cb-ref prompts);
    # the layer's remaining value is the printed overlap sweep + gated tiers

for name, df in (("labeled", labeled), ("test", test)):
    c = df[~df.grounded]
    print(f"{name}: closed-book matched {int(c.cb_hit.sum())}/{len(c)} | "
          f"containment {int(c.cb_contain.sum())} | pool-miss {int((c.cb_hit & c.cb_miss).sum())}")


bangla_math.parquet: q='Problem' a='Answer' -> +1205 questions
closed-book reference lookup: 1205 questions
labeled: closed-book matched 0/169 | containment 0 | pool-miss 0
test: closed-book matched 0/1155 | containment 0 | pool-miss 0


## 3_f: 
Encoder classifier on synthetic contrastive pairs
The benchmark's hallucinated answers are perturbations of correct ones (brief §3; BenHalluEval's
generation recipe). We rebuild that distribution offline from squad_bn: for every answerable
question, the gold answer is a faithful example and a *plausible perturbation* — same-paragraph
answer swap, same-article swap, single-digit jitter, global-pool swap — is a hallucinated one.
40% of pairs have their context dropped so the model learns the closed-book track too.

**Contamination guard (v5's QA-verifier lesson):** any squad_bn question whose normalized prompt
appears in the labeled sample is excluded from training, so the labeled-set AUC/gate below is an
honest estimate. Test-prompt overlap is *left in* deliberately — squad_bn is public external data
(allowed, brief §6) and the ref layer already exploits it.

Runs **before** the judges load; the encoder is freed afterwards. Skips gracefully if no
encoder model is available from the Hub or local cache.


In [15]:
# ---- v6: encoder classifier on synthetic squad_bn contrastive pairs ---------
labeled["p_clf"] = np.nan
test["p_clf"]    = np.nan
CLF_OK = False

EN_D, BN_D = "0123456789", "০১২৩৪৫৬৭৮৯"

if USE_CLF and tarball:
    rng = np.random.default_rng(42)
    labeled_prompts = set(labeled.p_norm)

    arts = []
    with tarfile.open(tarball, "r:bz2") as tf:
        for split in ("train", "validation"):
            with tf.extractfile(f"squad_bn/{split}.json") as fh:
                data = json.load(fh)
            for art in data["data"]:
                paras = []
                for para in art["paragraphs"]:
                    qas = []
                    for qa in para["qas"]:
                        answers = [a["text"] for a in qa.get("answers", []) if a.get("text")]
                        if answers and normalize(qa["question"]) not in labeled_prompts:
                            qas.append((qa["question"], answers))
                    if qas:
                        paras.append((para["context"], qas))
                if paras:
                    arts.append(paras)

    def digit_jitter(s):
        """Replace one digit (either script) with a different one - the 'year/number
        slightly altered' hallucination type the brief describes."""
        chars = list(str(s))
        idxs = [i for i, c in enumerate(chars) if c in EN_D or c in BN_D]
        if not idxs:
            return None
        i = int(rng.choice(idxs))
        pool = EN_D if chars[i] in EN_D else BN_D
        alts = [d for d in pool if d != chars[i]]
        chars[i] = str(rng.choice(alts))
        out = "".join(chars)
        return out if norm_ans(out) != norm_ans(s) else None

    global_pool = [qas[0][1][0] for paras in arts for _, qas in paras]

    rows = []
    for paras in arts:
        art_answers = [ans[0] for _, qas in paras for _, ans in qas]
        for ctx, qas in paras:
            para_answers = [ans[0] for _, ans in qas]
            for q, answers in qas:
                gold = answers[0]
                rows.append((ctx, q, gold, 1))
                neg = digit_jitter(gold) if rng.random() < 0.5 else None
                if neg is None:
                    cands = [a for a in para_answers if norm_ans(a) != norm_ans(gold)]
                    if not cands:
                        cands = [a for a in art_answers if norm_ans(a) != norm_ans(gold)]
                    if not cands:
                        cands = [global_pool[j] for j in rng.integers(0, len(global_pool), 8)
                                 if norm_ans(global_pool[j]) != norm_ans(gold)]
                    if cands:
                        neg = str(cands[int(rng.integers(0, len(cands)))])
                if neg and not contains_gold(neg, q, set(answers)):
                    rows.append((ctx, q, neg, 0))

    real_rows = list(globals().get("BH_CLF_ROWS", []))   # v6.2: real benchmark pairs first
    _kd = [(c, q, a, y) for (c, q, a, y) in globals().get("KD_CLF_ROWS", [])
           if normalize(q) not in labeled_prompts]   # v6.5: NCTB/MCQ pairs, labeled excluded
    if _kd:
        real_rows += _kd
        print(f"knowledge-dataset classifier pairs: +{len(_kd)} (NCTB + MCQ distractors)")
    # v6.3: uncapped same-shape training - keep ALL real QA pairs, then fill up to
    # CLF_MAX_TRAIN with synthetic squad_bn pairs (same QA distribution as the compass)
    n_syn = max(0, CLF_MAX_TRAIN - len(real_rows))
    order = rng.permutation(len(rows))[:n_syn]
    print(f"synthetic squad pool: {len(rows)} available, using {len(order)} "
          f"(cap {CLF_MAX_TRAIN}, {len(real_rows)} real pairs kept in full)")
    drop  = rng.random(len(order)) < CLF_CTX_DROP   # v6.21: context-drop share (0.0 = grounded-only)
    data_rows = real_rows + [(("" if d else rows[j][0]), rows[j][1], rows[j][2], rows[j][3])
                             for j, d in zip(order, drop)]
    data_rows = [data_rows[j] for j in rng.permutation(len(data_rows))]
    if real_rows:
        print(f"classifier data: {len(real_rows)} real BenHalluEval pairs + "
              f"{len(data_rows) - len(real_rows)} synthetic squad_bn pairs")
    n_pos = sum(1 for r in data_rows if r[3] == 1)
    print(f"synthetic training pairs: {len(data_rows)} ({n_pos} faithful / {len(data_rows)-n_pos} hallucinated)")

    from transformers import AutoTokenizer as _AT, AutoModelForSequenceClassification
    # v6.14: reuse a saved fine-tuned checkpoint when attached -> skips ~18 min of
    # retraining AND freezes p_clf (this training is nondeterministic across
    # transformers builds: closed AUC moved 0.551 -> 0.584 between two runs on the
    # same data, which is what actually moved the LB 0.843 -> 0.85).
    _CLF_SAVED = None
    for _r in ("/kaggle/input", WORK_DIR, "."):
        if not os.path.isdir(_r):
            continue
        for _dp, _dn, _fn in os.walk(_r):
            if os.path.basename(_dp) == CLF_SAVE_NAME and "config.json" in _fn:
                _CLF_SAVED = _dp
                break
        if _CLF_SAVED:
            break
    _clf_src = _CLF_SAVED or CLF_PATH
    clf_tok = _AT.from_pretrained(_clf_src)
    clf = AutoModelForSequenceClassification.from_pretrained(
        _clf_src, num_labels=2, ignore_mismatched_sizes=True).to("cuda:0")
    _EPOCHS = 0 if _CLF_SAVED else CLF_EPOCHS
    if _CLF_SAVED:
        print(f"classifier: loaded fine-tuned checkpoint from {_CLF_SAVED} -> TRAINING SKIPPED")

    try:   # v6.3: BanglaBERT preprocessing best practice (starter notebook)
        from normalizer import normalize as _bn_norm
        print("clf: using csebuetnlp normalizer")
    except Exception:
        _bn_norm = lambda s: s

    def clf_encode(ctxs, qs, rs):
        # question+response go in segment A so truncation prefers eating the context.
        # A is char-clipped (long rambling model answers in the training pairs made
        # only_second truncation throw), with a generic-truncation fallback for the
        # rare batch where A alone still exceeds max_length.
        first  = [_bn_norm(f"প্রশ্ন: {str(q)[:300]} উত্তর: {str(r)[:300]}")
                  for q, r in zip(qs, rs)]
        second = [(_bn_norm(str(c)[:1200]) if c else "কোনো অনুচ্ছেদ নেই") for c in ctxs]
        try:
            return clf_tok(first, second, truncation="only_second", max_length=CLF_MAX_LEN,
                           padding=True, return_tensors="pt")
        except Exception:
            return clf_tok(first, second, truncation=True, max_length=CLF_MAX_LEN,
                           padding=True, return_tensors="pt")

    @torch.inference_mode()
    def _clf_eval_probs(df):   # v6.3: light eval pass for compass checkpointing
        clf.eval()
        out = np.zeros(len(df))
        for lo in range(0, len(df), 64):
            part = df.iloc[lo:lo + 64]
            enc = clf_encode(part.context.tolist(), part.prompt_bn.tolist(),
                             part.response_bn.tolist())
            enc = {k: v.to("cuda:0") for k, v in enc.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                out[lo:lo + len(part)] = clf(**enc).logits.float().softmax(-1)[:, 1].cpu().numpy()
        clf.train()
        return out

    from sklearn.metrics import roc_auc_score as _auc
    best_auc, best_state = -1.0, None

    opt = torch.optim.AdamW(clf.parameters(), lr=CLF_LR, weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler()
    n_steps = math.ceil(len(data_rows) / CLF_BATCH) * _EPOCHS
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: max(0.0, 1.0 - s / max(1, n_steps)))
    clf.train()
    step = 0
    t0 = time.time()
    for ep in range(_EPOCHS):
        perm = rng.permutation(len(data_rows))
        running = []
        for lo in range(0, len(perm), CLF_BATCH):
            batch = [data_rows[j] for j in perm[lo:lo + CLF_BATCH]]
            enc = clf_encode([b[0] for b in batch], [b[1] for b in batch], [b[2] for b in batch])
            enc = {k: v.to("cuda:0") for k, v in enc.items()}
            y = torch.tensor([b[3] for b in batch], device="cuda:0")
            with torch.autocast("cuda", dtype=torch.float16):
                loss = torch.nn.functional.cross_entropy(clf(**enc).logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            running.append(float(loss)); step += 1
            if step % 100 == 0:
                print(f"  ep{ep} step {step}/{n_steps} loss {np.mean(running[-100:]):.4f} "
                      f"({(time.time()-t0)/60:.1f} min)")
        # v6.16: select the epoch on the GROUNDED AUC, not the all-rows AUC.
        # p_clf is only consumed where the meta-judge includes it, and that is the
        # grounded stack (coef 3.68 - the strongest feature there). The closed track
        # picks blend3 {p_t, p_q} and never reads p_clf, whose closed AUC is ~0.58
        # (chance). Selecting on the all-rows AUC therefore optimises a number the
        # pipeline mostly ignores: across two real runs it kept the classifier with
        # all=0.819/grounded=0.973 over all=0.809/grounded=0.978 - i.e. it chose the
        # WORSE grounded encoder because closed improved (0.551 -> 0.584), on a track
        # that discards p_clf entirely.
        _pv = _clf_eval_probs(labeled)
        _gm = labeled.grounded.values
        _all_auc = _auc(labeled.label.values, _pv)
        if _gm.sum() >= 20 and len(set(labeled.label.values[_gm])) == 2:
            ep_auc = _auc(labeled.label.values[_gm], _pv[_gm])
            _crit = "grounded"
        else:                       # degenerate split -> fall back to the old criterion
            ep_auc, _crit = _all_auc, "all"
        print(f"  epoch {ep} {_crit} AUC {ep_auc:.4f}  (all-rows {_all_auc:.4f})")
        if ep_auc > best_auc:
            best_auc = ep_auc
            best_state = {k: v.detach().cpu().clone() for k, v in clf.state_dict().items()}
    if best_state is not None:
        clf.load_state_dict(best_state)
        print(f"v6.3: kept epoch checkpoint with {_crit} AUC {best_auc:.4f}")
    clf.eval()
    if not _CLF_SAVED:
        try:
            _out = os.path.join(WORK_DIR, CLF_SAVE_NAME)
            clf.save_pretrained(_out); clf_tok.save_pretrained(_out)
            print(f"classifier: saved fine-tuned checkpoint -> {_out}  "
                  f"(attach as a Kaggle dataset to skip the ~18 min retrain and freeze p_clf)")
        except Exception as _e:
            print("classifier: save failed (non-fatal):", _e)

    @torch.inference_mode()
    def clf_scores(df, desc):
        out = np.zeros(len(df))
        for lo in tqdm(range(0, len(df), 64), desc=desc, leave=False):
            part = df.iloc[lo:lo + 64]
            enc = clf_encode(part.context.tolist(), part.prompt_bn.tolist(),
                             part.response_bn.tolist())
            enc = {k: v.to("cuda:0") for k, v in enc.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = clf(**enc).logits.float()
            out[lo:lo + len(part)] = logits.softmax(-1)[:, 1].cpu().numpy()
        return out

    from tqdm.auto import tqdm
    labeled["p_clf"] = clf_scores(labeled, "clf labeled")
    test["p_clf"]    = clf_scores(test, "clf test")
    CLF_OK = True

    from sklearn.metrics import roc_auc_score
    _g = labeled.grounded.values
    print(f"labeled AUC  all {roc_auc_score(labeled.label, labeled.p_clf):.3f} | "
          f"grounded {roc_auc_score(labeled.label[_g], labeled.p_clf[_g]):.3f} | "
          f"closed {roc_auc_score(labeled.label[~_g], labeled.p_clf[~_g]):.3f}")

    del clf
    for _v in ('opt', 'scaler'):
        if _v in dir():
            del globals()[_v]
    import gc; gc.collect(); torch.cuda.empty_cache()
    print("classifier member ready; encoder freed")
else:
    print("classifier member skipped:",
          "flag off" if not USE_CLF else "squad_bn tarball unavailable")


knowledge-dataset classifier pairs: +2799 (NCTB + MCQ distractors)
synthetic squad pool: 139285 available, using 34026 (cap 50000, 15974 real pairs kept in full)
classifier data: 15974 real BenHalluEval pairs + 34026 synthetic squad_bn pairs
synthetic training pairs: 50000 (24391 faithful / 25609 hallucinated)


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

classifier: saved fine-tuned checkpoint -> /kaggle/working/clf_finetuned  (attach as a Kaggle dataset to skip the ~18 min retrain and freeze p_clf)


clf labeled:   0%|          | 0/5 [00:00<?, ?it/s]

clf test:   0%|          | 0/40 [00:00<?, ?it/s]

labeled AUC  all 0.815 | grounded 0.976 | closed 0.553
classifier member ready; encoder freed
